# 🧮 24 · Generalised linear models for count data
### Poisson and negative-binomial regression, size factors, dispersion shrinkage and false-discovery control — the statistics inside DESeq2 and edgeR, built from scratch and checked against a known truth

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 26 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/26_glm_count_data.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`special`, `stats`, `optimize`), pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Level 1, especially [Module 05](05_likelihood_mle_map.ipynb) (likelihood, MLE), [Module 06](06_linear_regression_gwas.ipynb) (design matrices, p-values, Bonferroni) and [Module 09](09_logistic_regression.ipynb) (logistic regression); comfortable with NumPy and pandas |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [05 · Likelihood, MLE and MAP](05_likelihood_mle_map.ipynb), [06 · Linear regression](06_linear_regression_gwas.ipynb), [09 · Logistic regression](09_logistic_regression.ipynb), [21 · Bayesian learning](21_bayesian_learning_mcmc.ipynb), [23 · Model selection](23_model_selection.ipynb) |
| **Research frontier covered** | RNA-seq and single-cell differential expression (DESeq2/edgeR-style negative-binomial GLMs), false-discovery-rate control, pseudobulk analysis — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** Bulk RNA-seq counts come from `simulate_bulk_counts`, written in Section 1 (hidden truth:
> which genes change, their true log fold changes, dispersions and size factors); single-cell counts come from the
> toolkit's `simulate_scrna` (hidden truth: cell types, donors and the per-type stimulation effects `true_logfc`).
> Simulation lets us *grade* every method against the truth; real RNA-seq data add outliers, unknown confounders and
> no answer key — Section 9 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why sequencing produces counts whose variance grows with the mean, and **distinguish** Poisson (technical) from negative-binomial (biological) variation.
2. **Write** a generalised linear model as random component + linear predictor + link, **recognise** logistic, Poisson and negative-binomial regression as members of one family, and **use** offsets for library size.
3. **Derive** the Poisson and negative-binomial log-likelihoods, gradients and Hessians, and **implement** Newton's method / IRLS from scratch, vectorised over thousands of genes.
4. **Estimate** size factors with the median-of-ratios method and **demonstrate** the composition bias of total-count normalisation.
5. **Test** thousands of genes with Wald and likelihood-ratio tests, **control** the false discovery rate with Benjamini–Hochberg, and **measure** the realised FDR and power against the ground truth.
6. **Shrink** gene-wise dispersions toward a mean trend (empirical Bayes) and **show** why this rescues FDR control with three or four replicates.
7. **Analyse** single-cell data by pseudobulk and **explain** why tests that treat cells as replicates produce false discoveries (pseudoreplication).
8. **Build** design matrices with covariates and interactions, and **connect** each step to DESeq2, edgeR, limma-voom and glmGamPoi.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Counts are not Gaussian | reads as samples, Poisson vs negative binomial, a bulk simulator | 20 min |
| 2 · The GLM recipe | link, offset, Newton/IRLS from scratch, check against scikit-learn | 25 min |
| 3 · Overdispersion and the negative binomial | Gamma–Poisson mixture, NB likelihood, dispersion; 🧪 null p-values | 25 min |
| 4 · Normalisation: size factors | median of ratios; 🧪 composition bias | 15 min |
| 5 · Testing many genes | Wald, LRT, Bonferroni, Benjamini–Hochberg; 🧪 realised FDR and power | 25 min |
| 6 · Few replicates → shrink the dispersion | Cox–Reid, trend, empirical-Bayes MAP; 🧪 FDR rescued | 25 min |
| 7 · From bulk to single cells: pseudobulk | cell-level tests vs pseudobulk; 🧪 pseudoreplication | 20 min |
| 8 · Design matrices with covariates | batch as a covariate, a cell-type × stimulation interaction | 10 min |
| 9 · 🧭 From toy to real data | tools, resources, what goes wrong | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

*Which genes change their expression when cells are stimulated?* It is the most common question asked of RNA
sequencing. A typical bulk experiment treats a cell line with a drug (or knocks out a gene) and compares it with untreated
controls, with three or four replicate cultures per group. In a famous single-cell version, Kang *et al.* (2018) took
blood cells (PBMCs) from eight donors, split each sample into an untreated and an **interferon-β-stimulated** aliquot, and
sequenced thousands of cells per condition. Interferon is the body's antiviral alarm signal: within hours it switches on
a battery of "interferon-stimulated genes" (ISG15, MX1, IFIT1, …) in almost every immune cell, plus programmes specific
to particular cell types.

The statistics look easy — compare two group means for each gene — but four features of sequencing data make naïve
analyses fail:

1. **The data are counts** (how many sequenced fragments came from each gene), not continuous measurements, and
   their spread grows with their mean.
2. **Biological replicates disagree** much more than counting noise alone predicts, and with three replicates the
   amount of disagreement is itself hard to estimate.
3. **Samples are sequenced to different depths**, and a few very abundant genes can change the share of reads left
   for all the others.
4. **Tens of thousands of genes are tested at once**, so "p < 0.05" alone would produce hundreds of false discoveries.

This notebook builds, from scratch, the method that the field converged on — a **negative-binomial generalised linear
model (GLM)** with size factors, shared-information dispersion estimates and false-discovery-rate control, the core of
**DESeq2** and **edgeR** — and grades every step against a SIMULATED experiment whose truth we know. The last part
moves to single cells (a simulation mimicking the Kang design) and shows why the *donor*, not the cell, is the unit of
replication.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, special, stats
from sklearn.linear_model import PoissonRegressor

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`sample_negative_binomial` draws Gamma–Poisson counts, `benjamini_hochberg` turns p-values into FDR-adjusted q-values, `simulate_scrna` simulates a PBMC-like single-cell experiment with known truth (Section 7), `log_normalize` is the standard log(1 + scaled count) transform and `CLASS_COLORS` is the course palette. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def benjamini_hochberg(pvalues) -> np.ndarray:
    """Benjamini–Hochberg adjusted p-values (q-values): reject where q ≤ α to control the FDR at α."""
    p = np.asarray(pvalues, float)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty_like(p)
    out[order] = np.minimum(q, 1.0)
    return out


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


print("🧰 Toolkit loaded:", ", ".join(['sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'log_normalize', 'benjamini_hochberg']))

---
## 1 · Counts are not Gaussian

> 📍 **Where we are.** Linear regression ([Module 06](06_linear_regression_gwas.ipynb)) assumed Gaussian noise with a
> constant variance; logistic regression ([Module 09](09_logistic_regression.ipynb)) modelled 0/1 outcomes. Sequencing
> gives a third kind of outcome: non-negative integer **counts**. This section shows where the counts come from and how
> noisy they are.

### 🔬 Assay box — bulk RNA-seq

- **What is measured.** RNA is extracted from a population of cells (a tissue, a culture), converted to complementary
  DNA, cut into fragments and sequenced. Each sequenced fragment (a **read**) is mapped to the genome and counted
  towards the gene it came from. The result is a **count matrix**: genes × samples, each entry the number of reads.
- **Typical sizes.** About 20,000 protein-coding genes (60,000 annotated genes), 10–50 million reads per sample,
  3–6 biological replicates per condition in a typical lab experiment (hundreds of samples in consortia such as GTEx).
- **Typical artefacts.** Samples get different numbers of reads (*library size*); a few highly expressed genes take a
  large share of reads (*composition*); samples processed on different days differ (*batch*); longer genes and
  GC-rich fragments get more reads; occasional failed or swapped samples.

### Sequencing is sampling

Think of a library as a huge bag of fragments; gene $g$ makes up a fraction $p_g$ of the bag. Sequencing draws $N$
fragments at random. The count $Y_g$ is then binomial — $N$ draws, each "from gene $g$" with a small probability $p_g$ —
and all genes together are multinomial.

🧮 **Math refresher — the Poisson distribution.** When the number of draws $N$ is huge and each has a tiny probability
$p$, the binomial count is very well approximated by a **Poisson** distribution with rate $\lambda = Np$:

$$P(Y = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \qquad k = 0, 1, 2, \dots \qquad \mathbb E[Y] = \operatorname{Var}(Y) = \lambda .$$

*In words:* the chance of seeing exactly $k$ reads depends only on the expected number $\lambda$; the variance **equals**
the mean, so a gene expected at 100 reads typically lands within about $\pm 2\sqrt{100} = \pm 20$ of it. ($k!$ is
"$k$ factorial", $1 \times 2 \times \dots \times k$, and $0! = 1$.) The binomial variance $Np(1-p)$ differs from $Np$ only by
the factor $1 - p \approx 1$.

Let us check this by "re-sequencing" one library many times.

In [ ]:
# ── 1.1 Re-sequencing the same library: multinomial counts behave like Poisson counts (SIMULATED) ──
rng = np.random.default_rng(SEED + 1)
n_genes_lib = 2000
p_lib = np.exp(rng.normal(0, 1.6, n_genes_lib))
p_lib /= p_lib.sum()                                   # fractions of the library that each gene makes up
N_reads = 1_000_000
tech = rng.multinomial(N_reads, p_lib, size=200)       # 200 technical replicates (re-sequencing the same tube)

pick = np.argsort(p_lib)[[200, 1000, 1600, 1950, 1999]]   # genes from low to very high abundance
table = pd.DataFrame({"fraction p_g": p_lib[pick], "expected Np_g": N_reads * p_lib[pick],
                      "mean count": tech[:, pick].mean(0), "variance": tech[:, pick].var(0, ddof=1)})
table["variance / mean"] = table["variance"] / table["mean count"]
print("Technical replicates of ONE library (SIMULATED, 1,000,000 reads each):")
print(table.round(5).to_string())

🔍 **What just happened?** For genes spanning more than two orders of magnitude of expected count, the variance across
technical replicates is close to the mean (variance/mean ≈ 1; the small deviations are the sampling error of
estimating a variance from 200 replicates, and for the most abundant gene the binomial factor $1 - p_g$ starts to
matter). **Counting noise is Poisson.** Note what this implies: a gene with mean 10 has a standard deviation of about 3
(30 %), a gene with mean 10,000 one of about 100 (1 %) — low counts are intrinsically noisy, and no Gaussian model with a
single variance can describe all genes at once.

But real experiments do not re-sequence one tube. They compare **biological replicates** — different cultures,
animals or people — whose true expression levels differ. To study that, we need a simulator with a known truth.

### A simulated bulk experiment

The function below plays the role of a laboratory. It decides the truth (abundances, which 10 % of genes change and by
how much), shares a fixed read budget among the genes of each sample, and adds biological variability with the
negative-binomial distribution, which Section 3 derives. Genes are rows and samples are columns, as in DESeq2 and edgeR
(single-cell tools, [Module 30](30_single_cell_rnaseq.ipynb), put cells in rows).

In [ ]:
# ── 1.2 A simulated bulk RNA-seq experiment with a hidden truth ──
def simulate_bulk_counts(rng, n_genes=2000, n_per_group=4, frac_de=0.1, depth=1e6, n_dominant=0, dominant_fold=20.0,
                         batch_effect=0.0, disp_asymptote=0.03, disp_extra=1.0, disp_sd=0.5):
    """SIMULATED bulk RNA-seq, control vs treated; returns genes × samples counts and the hidden truth.

    Gene g has abundance q_g (log-normal); a fraction `frac_de` of genes change by a true log2 fold change in treated
    samples; `n_dominant` abundant genes can be induced `dominant_fold`-fold (Section 4). Sample i is sequenced to
    depth D_i and the read budget is shared: mu_gi = D_i q_gi / sum_g q_gi. Replicates scatter around mu_gi with
    negative-binomial noise of dispersion alpha_g = (disp_asymptote + disp_extra / mean_g) * exp(N(0, disp_sd²)).
    `batch_effect` (sd, log scale) shifts every gene in batch 2, which holds half of each group.
    """
    n = 2 * n_per_group
    treated = np.repeat([0, 1], n_per_group)
    batch = np.tile([0, 1], n_per_group)
    log_q = rng.normal(0.0, 1.6, n_genes)
    lfc2 = np.zeros(n_genes)
    de = rng.choice(n_genes, int(round(frac_de * n_genes)), replace=False)
    lfc2[de] = rng.choice([-1, 1], len(de)) * rng.uniform(0.4, 2.0, len(de))
    if n_dominant:                                         # abundant genes that become huge after treatment
        top = np.setdiff1d(np.argsort(log_q)[-200:], de)
        lfc2[rng.choice(top, n_dominant, replace=False)] = np.log2(dominant_fold)
    log_abund = log_q[:, None] + np.log(2) * lfc2[:, None] * treated[None, :]
    log_abund = log_abund + batch_effect * rng.normal(0, 1, (n_genes, 1)) * batch[None, :]
    abund = np.exp(log_abund)
    D = depth * np.exp(rng.normal(0, 0.3, n))              # sequencing depth differs between samples
    mu = D * abund / abund.sum(axis=0)                      # a fixed read budget per sample: composition!
    sf = D / abund.sum(axis=0)
    sf = sf / np.exp(np.log(sf).mean())                     # true size factors (geometric mean 1)
    base_mean = (mu / sf)[:, treated == 0].mean(axis=1)
    alpha = (disp_asymptote + disp_extra / base_mean) * np.exp(rng.normal(0, disp_sd, n_genes))
    Y = sample_negative_binomial(rng, mu, 1 / alpha[:, None])
    gnames = [f"gene{g:04d}" for g in range(n_genes)]
    snames = [("ctrl" if t == 0 else "trt") + str(i % n_per_group + 1) for i, t in enumerate(treated)]
    samples = pd.DataFrame({"condition": np.where(treated == 1, "treated", "control"), "treated": treated,
                            "batch": batch, "true_size_factor": sf}, index=snames)
    genes = pd.DataFrame({"base_mean": base_mean, "true_alpha": alpha, "true_log2fc": lfc2, "is_de": lfc2 != 0},
                         index=gnames)
    return dict(counts=pd.DataFrame(Y, index=gnames, columns=snames), samples=samples, genes=genes)

In [ ]:
# ── 1.3 Our experiment: 4 control vs 4 treated cultures (SIMULATED) ──
bulk = simulate_bulk_counts(np.random.default_rng(SEED + 2))
counts, samples, truth = bulk["counts"], bulk["samples"], bulk["genes"]
print("SIMULATED bulk RNA-seq:", counts.shape[0], "genes ×", counts.shape[1], "samples;",
      int(truth.is_de.sum()), "truly changed genes (hidden truth)")
print(counts.head(4).to_string())
print("\nlibrary sizes (millions of reads):", (counts.sum(axis=0) / 1e6).round(2).tolist())
print("true size factors:              ", samples.true_size_factor.round(2).tolist())
print("expression range (base mean, 1st/50th/99th percentile):", truth.base_mean.quantile([.01, .5, .99]).round(1).tolist())

🔍 **What just happened?** We have a count matrix of 2,000 genes × 8 samples (4 control, 4 treated) — about one tenth
of a real transcriptome, which keeps every computation below to seconds. The library sizes differ almost three-fold
(0.55 to 1.55 million reads), as they do in real experiments, and the genes span more than three orders of magnitude of
expression. Hidden in `truth` are the 200
genes that really change; a good method should find them and (almost) nothing else.

### Technical versus biological replicates

Now the key picture: variance against mean for technical replicates (re-sequencing) and for biological replicates
(different cultures). We simulate 10 replicates of each so that the variance of every gene is estimated reasonably well.
The counts are divided by their size factors first, so that differences in depth do not add spurious variance.

In [ ]:
# ── 1.4 Mean–variance relationship: technical (Poisson) vs biological (negative binomial) replicates ──
rich = simulate_bulk_counts(np.random.default_rng(SEED + 3), n_per_group=10)
ctrl = rich["samples"].condition.values == "control"
norm_bio = rich["counts"].values[:, ctrl] / rich["samples"].true_size_factor.values[ctrl]
expected = norm_bio.mean(axis=1)
tech10 = np.random.default_rng(SEED + 4).poisson(expected[:, None], size=(len(expected), 10))  # re-sequencing

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
grid = np.logspace(-1, 4.5, 100)
for ax, data, title in [(axes[0], tech10, "technical replicates (re-sequencing)"),
                        (axes[1], norm_bio, "biological replicates (different cultures)")]:
    m, v = data.mean(1), data.var(1, ddof=1)
    ok = m > 0
    ax.scatter(m[ok], v[ok] + 1e-2, s=4, alpha=0.4, color="0.4")
    ax.plot(grid, grid, color=CLASS_COLORS[0], lw=2, label="Poisson: variance = mean")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(0.3, 3e4); ax.set_ylim(1e-1, 1e7)
    ax.set_xlabel("mean normalised count"); ax.set_title(f"SIMULATED {title}")
a_trend = 0.03 + 1.0 / grid                                   # the simulator's dispersion trend (no gene scatter)
axes[1].plot(grid, grid + a_trend * grid ** 2, color=CLASS_COLORS[4], lw=2, label=r"NB: mean + $\alpha$·mean$^2$")
axes[0].set_ylabel("variance across 10 replicates")
for ax in axes:
    ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

m, v = norm_bio.mean(1), norm_bio.var(1, ddof=1)
hi = m > 1000
print(f"genes with mean > 1000: median variance/mean = {np.median(v[hi] / m[hi]):.0f} (biological) vs "
      f"{np.median(tech10.var(1, ddof=1)[hi] / tech10.mean(1)[hi]):.2f} (technical)")

🔍 **What just happened?** The technical replicates hug the Poisson line at every expression level. The biological
replicates look similar only for weakly expressed genes (means below about 10); above that the variance bends upwards
along a **parabola**, $\text{mean} + \alpha\,\text{mean}^2$, and for well-expressed genes it is tens of times larger than Poisson
(the printed median ratio). The extra variance is the cultures' genuine disagreement about how much of each gene to
make. This **overdispersion** is the single most important fact about RNA-seq counts: a test that assumes Poisson
noise treats ordinary biological variability as a treatment effect (Section 3 measures how badly).

⚠️ **Pitfall: "just log-transform and use a t-test".** It is tempting to take $\log(y + 1)$ of normalised counts and use
the Gaussian tools of Module 06. With many replicates this works reasonably (limma-voom, Section 9, does a weighted
version of it), but with 3–4 replicates the per-gene variances are very unreliable, the log of small counts is distorted
by the "+1", and the mean–variance trend is ignored. Count models use what we know about the noise.

✏️ **Your turn.** In cell 1.4, change `disp_asymptote` in the call to `simulate_bulk_counts` (add
`disp_asymptote=0.2`). Where does the biological cloud leave the Poisson line now? What kind of experiment might have
such large biological variability — inbred mice, cell lines or human patients?

---
## 2 · The GLM recipe

> 📍 **Where we are.** We know *what* the noise looks like. Now we need a regression model that (i) produces
> non-negative means, (ii) lets the variance depend on the mean and (iii) handles samples of different depth. The answer
> is a **generalised linear model** (Nelder & Wedderburn 1972) — and you have already met one.

A GLM has three ingredients:

1. a **random component** — the distribution of $y_i$ given its mean $\mu_i$ (Gaussian, Bernoulli, Poisson, negative
   binomial, …);
2. a **linear predictor** $\eta_i = \mathbf x_i^\top\boldsymbol\beta$ — the same design-matrix machinery as Module 06;
3. a **link function** $g$ connecting them: $g(\mu_i) = \eta_i$.

| model | random component | link $g(\mu)$ | variance as a function of the mean |
|---|---|---|---|
| linear regression (Module 06) | Gaussian | identity, $\mu$ | $\sigma^2$ (constant) |
| logistic regression (Module 09) | Bernoulli | logit, $\log\frac{\mu}{1-\mu}$ | $\mu(1-\mu)$ |
| Poisson regression | Poisson | log, $\log\mu$ | $\mu$ |
| negative-binomial regression | negative binomial | log, $\log\mu$ | $\mu + \alpha\mu^2$ |

> 🔗 **Builds on** [Module 09 · Logistic regression](09_logistic_regression.ipynb), Section on the gradient: there you
> found $\partial \ell/\partial \mathbf w = \sum_\ell \mathbf x^\ell (y^\ell - \hat p^\ell)$ — features times prediction
> errors. We are about to derive *exactly the same formula* for counts. That is no coincidence: for every GLM with its
> "canonical" link the gradient is $X^\top(\mathbf y - \boldsymbol\mu)$.

### The model for one gene

For gene $g$ in sample $i$, with $x_i = 1$ for treated and $0$ for control samples:

$$y_{gi} \sim \text{Poisson}(\mu_{gi}), \qquad \log \mu_{gi} = \underbrace{\log s_i}_{\text{offset}} + \beta_{g0} + \beta_{g1} x_i .$$

*In words:* the expected count is the sample's size factor $s_i$ times a gene-specific expression level
$e^{\beta_{g0}}$, multiplied by $e^{\beta_{g1}}$ in treated samples. So $\beta_{g1}$ is the **log fold change** (natural
log); DESeq2 reports $\log_2$ fold changes, $\beta_{g1}/\ln 2$. A log2 fold change of $-1$ means "half as much".

The **offset** $\log s_i$ is a column of the design matrix whose coefficient is *fixed at 1*: a sample sequenced twice
as deeply has, all else equal, exactly twice the expected count. If we instead put $\log s_i$ in as an ordinary
covariate, the model would estimate its coefficient from 8 samples — wasting a degree of freedom on something we
already know and adding noise.

### Likelihood, gradient and Hessian

With $\eta_i = \log s_i + \mathbf x_i^\top\boldsymbol\beta$ (now $\mathbf x_i = (1, x_i)$) and $\mu_i = e^{\eta_i}$, the Poisson
log-likelihood of one gene is

$$\ell(\boldsymbol\beta) = \sum_i \big[y_i\eta_i - e^{\eta_i} - \log y_i!\big], \qquad
\nabla\ell = \sum_i (y_i - \mu_i)\,\mathbf x_i = X^\top(\mathbf y - \boldsymbol\mu), \qquad
\nabla^2\ell = -\sum_i \mu_i\,\mathbf x_i\mathbf x_i^\top = -X^\top W X ,$$

with $W = \operatorname{diag}(\mu_1, \dots, \mu_n)$. *In words:* the gradient is again "features × (observed − expected)";
the Hessian (the matrix of second derivatives, [Module 04](04_calculus_and_optimization.ipynb)) is minus a weighted
$X^\top X$, which is negative definite — the log-likelihood is a single smooth hill with one top (concave, "Mount Fuji").

🧮 **Math refresher — Newton's method.** Gradient ascent climbs with a fixed step size. Newton's method also uses the
*curvature*: near the top a concave function looks like an upside-down parabola, and Newton jumps straight to that
parabola's peak. In one dimension $\beta \leftarrow \beta - \ell'(\beta)/\ell''(\beta)$; in several,

$$\boldsymbol\beta \leftarrow \boldsymbol\beta + (X^\top W X)^{-1} X^\top(\mathbf y - \boldsymbol\mu) .$$

*In words:* move along the gradient, but rescaled by the inverse curvature — big steps where the hill is flat, small
steps where it is steep. Close to the optimum the number of correct digits roughly doubles at every step.

Rearranging gives the classic name. Define the **working response** $z_i = \eta_i - \log s_i + (y_i - \mu_i)/\mu_i$; then
the Newton step is $\boldsymbol\beta \leftarrow (X^\top W X)^{-1}X^\top W\mathbf z$ — a **weighted least-squares** fit
(the normal equations of Module 06 with weights). Repeating it is **iteratively reweighted least squares (IRLS)**, the
algorithm R's `glm()` and DESeq2 use.

In [ ]:
# ── 2.1 Newton/IRLS for one gene, step by step ──
def irls_poisson_one(y, X, offset, n_iter=8):
    """Poisson GLM for a single gene by IRLS; prints the climb. Returns beta."""
    beta = np.array([np.log(np.mean(y / np.exp(offset)) + 0.1), 0.0])       # start: overall level, no change
    for it in range(n_iter):
        eta = X @ beta + offset
        mu = np.exp(eta)
        loglik = np.sum(y * eta - mu - special.gammaln(y + 1))
        print(f"  iter {it}: beta = ({beta[0]:8.5f}, {beta[1]:8.5f})   log-likelihood = {loglik:.6f}")
        z = eta - offset + (y - mu) / mu                                    # working response
        W = mu                                                               # weights = variance of each count
        beta = np.linalg.solve(X.T @ (W[:, None] * X), X.T @ (W * z))         # weighted least squares
    return beta


treated = samples.treated.values.astype(float)
X = np.column_stack([np.ones_like(treated), treated])         # design matrix: intercept, treated indicator
s_true = samples.true_size_factor.values
g_demo = truth.index[(truth.is_de) & (truth.base_mean > 200)][0]
y = counts.loc[g_demo].values.astype(float)
print(f"{g_demo}: counts {y.astype(int).tolist()}   (true log2 fold change {truth.loc[g_demo, 'true_log2fc']:.2f})")
beta = irls_poisson_one(y, X, np.log(s_true))

# For two groups the Poisson MLE has a closed form: exp(beta0) = sum(y)/sum(s) in the control group, and so on
closed = np.log([y[treated == 0].sum() / s_true[treated == 0].sum(), y[treated == 1].sum() / s_true[treated == 1].sum()])
print(f"closed form: beta0 = {closed[0]:.5f}, beta0 + beta1 = {closed[1]:.5f};  IRLS: {beta[0]:.5f}, {beta.sum():.5f}")
print(f"estimated log2 fold change = beta1 / ln 2 = {beta[1] / np.log(2):.3f}")

In [ ]:
# ── 2.2 Check against scikit-learn ──
# PoissonRegressor has no offset argument, but for the Poisson likelihood an offset log(s) is equivalent to fitting
# the RATE y/s with sample weights s:  sum_i s_i [(y_i/s_i) eta_i - e^eta_i] = sum_i [y_i eta_i - s_i e^eta_i].
skl = PoissonRegressor(alpha=0.0, tol=1e-10, max_iter=1000)
skl.fit(treated.reshape(-1, 1), y / s_true, sample_weight=s_true)
print(f"scikit-learn: beta0 = {skl.intercept_:.5f}, beta1 = {skl.coef_[0]:.5f}")
print(f"our IRLS:     beta0 = {beta[0]:.5f}, beta1 = {beta[1]:.5f}")

🔍 **What just happened?** Starting from "no change", IRLS climbed to the maximum in a handful of iterations: the
log-likelihood increases at every step and then stops changing in the printed digits — Newton's quadratic convergence.
The answer equals the two-group closed form (total count divided by total size factor in each group) and scikit-learn's
optimiser to the printed precision. The estimated log2 fold change is close to the true one; the remaining difference
is biological noise in four replicates per group.

Real analyses fit 20,000 genes, so we now vectorise: all genes share the design matrix $X$, and only the weights
differ. The function below fits every gene at once. It already accepts a dispersion `alpha` for Section 3 (with
`alpha = 0` it is the Poisson model). Each gene needs its own $2 \times 2$ matrix $X^\top W_g X$; `np.einsum` builds all of
them in one line and `np.linalg.solve` solves all the small systems together.

In [ ]:
# ── 2.3 IRLS vectorised over genes (Poisson now, negative binomial in Section 3) ──
def glm_irls(Y, X, offset, alpha=0.0, n_iter=30, tol=1e-8):
    """Fit log-link count GLMs to every row (gene) of Y (genes × samples) with design X and offset log(s).

    alpha: NB dispersion per gene (0 = Poisson). Returns beta (genes × p), standard errors (genes × p),
    fitted means mu (genes × samples) and the number of genes that have not converged.
    """
    Y = np.asarray(Y, float)
    a = np.broadcast_to(np.asarray(alpha, float), (Y.shape[0],))[:, None]
    beta = np.zeros((Y.shape[0], X.shape[1]))
    beta[:, 0] = np.log((Y / np.exp(offset)).mean(1) + 0.1)
    for _ in range(n_iter):
        eta = np.clip(beta @ X.T + offset, -30, 30)
        mu = np.exp(eta)
        W = mu / (1 + a * mu)                                   # IRLS weights (= mu for Poisson)
        z = eta - offset + (Y - mu) / mu                        # working response
        XtWX = np.einsum("np,gn,nq->gpq", X, W, X)              # one p × p matrix per gene
        XtWz = np.einsum("np,gn->gp", X, W * z)
        new = np.linalg.solve(XtWX, XtWz[..., None])[..., 0]
        change = np.abs(new - beta).max(axis=1)
        beta = new
        if change.max() < tol:
            break
    mu = np.exp(np.clip(beta @ X.T + offset, -30, 30))
    W = mu / (1 + a * mu)
    cov = np.linalg.inv(np.einsum("np,gn,nq->gpq", X, W, X))    # inverse Fisher information
    se = np.sqrt(np.einsum("gpp->gp", cov))
    return beta, se, mu, int((change > 1e-4).sum())


Y = counts.values
t0 = time.time()
b_pois, se_pois, mu_pois, n_bad = glm_irls(Y, X, np.log(s_true))
print(f"Poisson GLM for {len(Y)} genes in {time.time() - t0:.2f} s; genes not converged: {n_bad}")
print(f"gene {g_demo}: vectorised beta = {b_pois[truth.index.get_loc(g_demo)].round(5)}  (cell 2.1: {beta.round(5)})")

In [ ]:
# ── 2.4 🧪 Ground-truth check: are the estimated fold changes right? ──
de = truth.is_de.values
est_l2fc = b_pois[:, 1] / np.log(2)
print(f"🧪 truly changed genes: correlation(estimated, true log2FC) = {np.corrcoef(est_l2fc[de], truth.true_log2fc[de])[0, 1]:.3f}; "
      f"median |error| = {np.median(np.abs(est_l2fc[de] - truth.true_log2fc[de])):.3f}")
print(f"🧪 unchanged genes: median |estimated log2FC| = {np.median(np.abs(est_l2fc[~de])):.3f}")

🔍 **What just happened?** All 2,000 genes were fitted in a fraction of a second, and the demo gene's coefficients equal
the one-gene fit. Every gene converged here, but beware: when every count of one group is zero, the MLE of that group's
log mean is $-\infty$ — the count version of the *separable data* problem of Module 09 — and IRLS keeps stepping
towards it (our function reports such genes). Production tools cap these estimates or add a weak prior (DESeq2's
fold-change shrinkage, Section 6).

The 🧪 check shows that the Poisson GLM estimates fold changes well: the correlation with the truth is high and the
typical error is about 0.17 log2 units — about the same size as the spurious "fold changes" of the unchanged genes,
which is the biological noise of four replicates. **The point estimates are not the problem — the uncertainty is.** The standard error
$\sqrt{[(X^\top WX)^{-1}]_{11}}$ assumes Poisson noise, and we saw in Section 1 that biological replicates are far noisier.

✏️ **Your turn.** Remove the offset (pass `np.zeros(8)` instead of `np.log(s_true)`) and re-run cells 2.3–2.4. How does the
median |estimated log2FC| of the unchanged genes change? Which samples' library sizes are responsible?

---
## 3 · Overdispersion and the negative binomial

> 📍 **Where we are.** The Poisson GLM gives sensible fold changes but assumes variance = mean. We now build a count
> distribution whose variance can be larger, derive its likelihood, and measure what the wrong noise model does to
> p-values — the first 🧪 ground-truth check on null genes.

### A two-step story

Each biological replicate has its *own* true expression rate $\lambda_i$, scattered around the gene's mean $\mu$; the
sequencer then counts Poisson reads around $\lambda_i$:

$$\lambda_i \sim \text{Gamma}(\text{shape} = \theta,\ \text{mean} = \mu), \qquad y_i \mid \lambda_i \sim \text{Poisson}(\lambda_i) .$$

*In words:* first nature picks how much the culture really expresses the gene, then sequencing counts it.

🧮 **Math refresher — the Gamma distribution and the law of total variance.** The Gamma distribution describes a
positive quantity; with shape $\theta$ and mean $\mu$ its variance is $\mu^2/\theta$ (large $\theta$ = tightly
concentrated). For a two-step model, the **law of total variance** says
$\operatorname{Var}(Y) = \mathbb E[\operatorname{Var}(Y\mid\lambda)] + \operatorname{Var}(\mathbb E[Y\mid\lambda])$ — noise within a step
plus variation of the step's mean. Here:

$$\operatorname{Var}(Y) = \mathbb E[\lambda] + \operatorname{Var}(\lambda) = \mu + \frac{\mu^2}{\theta} = \mu + \alpha\mu^2, \qquad \alpha = 1/\theta .$$

*In words:* Poisson counting noise ($\mu$) plus biological noise that grows with the square of the mean
($\alpha\mu^2$). $\alpha$ is the **dispersion** (DESeq2's term); $\sqrt\alpha$ is the biological **coefficient of variation**
(edgeR's "BCV") — $\alpha = 0.04$ means replicates differ by about 20 % in true expression. As $\alpha \to 0$ we recover
the Poisson.

Averaging the Poisson over the Gamma gives a closed form, the **negative binomial (NB)** distribution:

$$P(y \mid \mu, \theta) = \frac{\Gamma(y + \theta)}{\Gamma(\theta)\, y!}\left(\frac{\theta}{\theta + \mu}\right)^{\theta}\left(\frac{\mu}{\theta + \mu}\right)^{y} .$$

*In words:* the probability of a count $y$ given a mean $\mu$ and a "shape" $\theta$; $\Gamma$ is the **Gamma function**, a
smooth version of the factorial ($\Gamma(n) = (n-1)!$ for whole numbers). In code we always use its logarithm
`scipy.special.gammaln`, because $\Gamma(1000)$ overflows any computer.

In [ ]:
# ── 3.1 The Gamma–Poisson mixture IS the negative binomial (SIMULATED) ──
rng = np.random.default_rng(SEED + 5)
mu0, theta0 = 20.0, 3.0
lam = rng.gamma(theta0, mu0 / theta0, 100_000)            # step 1: each replicate's true rate
ymix = rng.poisson(lam)                                     # step 2: Poisson counting
k = np.arange(0, 90)
nb_pmf = stats.nbinom.pmf(k, n=theta0, p=theta0 / (theta0 + mu0))   # SciPy's parameterisation (n = theta)
print(f"mixture: mean {ymix.mean():.2f}, variance {ymix.var():.1f};  formula mu + mu²/theta = {mu0 + mu0**2 / theta0:.1f}")

plt.figure(figsize=(7, 3.4))
plt.hist(ymix, bins=np.arange(-0.5, 90), density=True, color="0.75", label="Gamma–Poisson draws (SIMULATED)")
plt.plot(k, nb_pmf, color=CLASS_COLORS[4], lw=2, label=r"negative binomial pmf ($\mu$ = 20, $\theta$ = 3)")
plt.plot(k, stats.poisson.pmf(k, mu0), color=CLASS_COLORS[0], lw=2, ls="--", label="Poisson pmf with the same mean")
plt.xlabel("count y"); plt.ylabel("probability")
plt.title("Same mean, very different spread: the mixture is negative binomial")
plt.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** The histogram of 100,000 two-step draws sits exactly on the NB formula, and its variance
matches $\mu + \mu^2/\theta$. The Poisson with the same mean (dashed) is far too narrow: counts of 0–5 or above 50 are
plausible under the NB but essentially impossible under the Poisson. A test built on the Poisson would call such
ordinary replicates "significantly different".

### The NB GLM

Replace the Poisson by the NB in the GLM, keep the log link and the offset. For a known $\alpha$ the gradient becomes
$X^\top\big[(\mathbf y - \boldsymbol\mu)/(1 + \alpha\boldsymbol\mu)\big]$ and the expected curvature (Fisher information) is
$X^\top W X$ with $W = \operatorname{diag}\big(\mu_i/(1 + \alpha\mu_i)\big)$. *In words:* the same IRLS as before, but
high-count samples get less weight, because their variance is inflated by the biological term. That is the one-line
change already built into `glm_irls`.

The dispersion itself is estimated by maximising the NB log-likelihood over $\alpha$ with the means held at their fitted
values; we alternate "fit $\boldsymbol\beta$ given $\alpha$" and "fit $\alpha$ given $\boldsymbol\mu$" a few times.

In [ ]:
# ── 3.2 The NB log-likelihood, and a vectorised one-dimensional optimiser for the dispersion ──
def nb_loglik(Y, mu, alpha):
    """NB log-likelihood summed over samples, one value per gene (alpha = dispersion = 1/theta, one per gene)."""
    Y = np.asarray(Y, float)
    th = 1.0 / np.asarray(alpha, float)[:, None]
    return (special.gammaln(Y + th) - special.gammaln(th) - special.gammaln(Y + 1)
            + th * np.log(th / (th + mu)) + Y * np.log(mu / (th + mu))).sum(axis=1)


def golden_max(f, lo, hi, n_genes, n_iter=50):
    """Maximise a vectorised function f(x) -> one value per gene over x in [lo, hi] by golden-section search."""
    a, b = np.full(n_genes, lo, float), np.full(n_genes, hi, float)
    r = (np.sqrt(5) - 1) / 2
    for _ in range(n_iter):                              # each step keeps the better 61.8 % of the interval
        c, d = b - r * (b - a), a + r * (b - a)
        left = f(c) > f(d)
        b, a = np.where(left, d, b), np.where(left, a, c)
    return (a + b) / 2


def fit_dispersion_ml(Y, mu):
    """Gene-wise maximum-likelihood dispersion given fitted means (searched on the log scale, 1e-8 … 10)."""
    return np.exp(golden_max(lambda la: nb_loglik(Y, mu, np.exp(la)), np.log(1e-8), np.log(10.0), len(Y)))


# check nb_loglik against SciPy for one gene
g0 = 7
ref = stats.nbinom.logpmf(Y[g0], n=1 / 0.05, p=(1 / 0.05) / (1 / 0.05 + mu_pois[g0])).sum()
print(f"nb_loglik = {nb_loglik(Y[g0:g0 + 1], mu_pois[g0:g0 + 1], np.array([0.05]))[0]:.6f};  SciPy: {ref:.6f}")

In [ ]:
# ── 3.3 Fit the NB GLM to every gene: alternate beta | alpha and alpha | mu ──
offset = np.log(s_true)
alpha_ml = np.full(len(Y), 0.1)
t0 = time.time()
for rnd in range(3):
    b_nb, se_nb, mu_nb, _ = glm_irls(Y, X, offset, alpha_ml)
    alpha_ml = fit_dispersion_ml(Y, mu_nb)
b_nb, se_nb, mu_nb, n_bad = glm_irls(Y, X, offset, alpha_ml)
print(f"NB GLM with gene-wise ML dispersions: {time.time() - t0:.2f} s")

# one gene, checked with SciPy's scalar optimiser
g1 = 11
res = optimize.minimize_scalar(lambda la: -nb_loglik(Y[g1:g1 + 1], mu_nb[g1:g1 + 1], np.exp([la]))[0],
                               bounds=(np.log(1e-8), np.log(10.0)), method="bounded")
print(f"gene {g1}: golden section alpha = {alpha_ml[g1]:.5f};  scipy minimize_scalar alpha = {np.exp(res.x):.5f}")

true_alpha = truth.true_alpha.values
print(f"median true alpha = {np.median(true_alpha):.4f};  median estimated = {np.median(alpha_ml):.4f};  "
      f"median ratio estimate/truth = {np.median(alpha_ml / true_alpha):.2f}")
print(f"genes whose estimate hit the lower bound (sample variance below Poisson): {np.mean(alpha_ml < 1e-6):.1%}")

🔍 **What just happened?** Our vectorised golden-section search agrees with SciPy's `minimize_scalar` on a test gene,
and the whole NB fit took well under a second. But look at the accuracy: the gene-wise **maximum-likelihood dispersions
are biased low** — the median estimate is only about 57 % of the true value — and for 10 % of the genes the four-plus-four
replicates happened to agree so well that the best $\alpha$ is (essentially) zero. Remember why the MLE of a variance
divides by $N$ instead of $N - 1$ ([Module 05](05_likelihood_mle_map.ipynb)): it ignores that the mean was estimated from
the same data. Here two coefficients were estimated from eight samples, so the bias is substantial. We come back to
this in Section 6.

### 🧪 Ground-truth check: p-values of genes that do not change

For the 1,800 truly unchanged genes, a valid test must give **uniformly distributed** p-values: 5 % below 0.05, 1 %
below 0.01. We compute the **Wald test** (estimate divided by its standard error, compared with a standard normal — the
same idea as the $t$-statistics of Module 06; Section 5 gives details) under three noise models: Poisson, NB with the
*true* dispersions (an oracle we never have in practice), and NB with the gene-wise ML estimates.

In [ ]:
# ── 3.4 🧪 Null p-values under three noise models ──
def wald_pvalues(beta, se, j=1):
    """Two-sided Wald p-values for coefficient j."""
    return 2 * stats.norm.sf(np.abs(beta[:, j] / se[:, j]))


b_or, se_or, mu_or, _ = glm_irls(Y, X, offset, true_alpha)            # the oracle: true dispersions
pvals = {"Poisson GLM": wald_pvalues(b_pois, se_pois),
         "NB GLM, true dispersions (oracle)": wald_pvalues(b_or, se_or),
         "NB GLM, gene-wise ML dispersions": wald_pvalues(b_nb, se_nb)}
null = ~truth.is_de.values

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, (name, p) in zip(axes, pvals.items()):
    ax.hist(p[null], bins=20, range=(0, 1), color="0.6", edgecolor="w")
    ax.axhline(null.sum() / 20, color=CLASS_COLORS[1], ls="--", lw=1.5, label="uniform (valid test)")
    ax.set_title(f"{name}\n{np.mean(p[null] < 0.05):.1%} of null genes have p < 0.05")
    ax.set_xlabel("p-value (truly unchanged genes)")
axes[0].set_ylabel("number of genes"); axes[0].legend()
plt.suptitle("SIMULATED null genes: only the right noise model gives uniform p-values", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** This is the central lesson of the notebook in one figure.

- **Poisson:** almost half of the genes that do *not* change get $p < 0.05$ — a pile-up at zero instead of a flat
  histogram. The Poisson standard errors are far too small because they ignore biological variability, so ordinary
  replicate-to-replicate differences look like treatment effects. Poisson p-values are **anti-conservative**.
- **NB with the true dispersions:** the histogram is flat and 5.7 % of null genes fall below 0.05 — within sampling
  error of 5 %. With the right noise model, the Wald test does what it promises.
- **NB with estimated dispersions:** much better than Poisson, but still too many small p-values (14 % below 0.05). Genes whose dispersion is *underestimated* — by chance or because of the downward
  bias — get standard errors that are too small. With 4 + 4 samples a gene-wise dispersion is simply hard to estimate.

So the noise model must not only be the right *family* (NB), its parameter must also be estimated well. Sections 5–6
show what this means for a whole experiment and how DESeq2 and edgeR solve it — by sharing information across genes.

✏️ **Your turn.** Re-run cells 3.3–3.4 on a richer experiment: in cell 1.3 change `simulate_bulk_counts(np.random.default_rng(SEED + 2))`
to `simulate_bulk_counts(np.random.default_rng(SEED + 2), n_per_group=10)` and run cells 1.3 → 3.4 again. How close to
5 % do the gene-wise ML dispersions get with ten replicates per group? (Change it back afterwards.)

---
## 4 · Normalisation: size factors

> 📍 **Where we are.** So far we cheated: we used the *true* size factors $s_i$. Real analyses must estimate them from
> the counts. This section compares the obvious estimate with DESeq2's and shows a trap that catches the obvious one.

**Total-count scaling** (counts per million, CPM) sets $s_i \propto \sum_g y_{gi}$, the library size. It corrects for
depth — but sequencing is a *fixed read budget*: if a few genes become hugely expressed in treated samples, they take a
larger share of the reads and **every other gene gets fewer reads**, even though its expression per cell did not
change. Total-count scaling then makes thousands of unchanged genes look down-regulated. This is **composition bias**
(Robinson & Oshlack 2010 introduced the TMM correction for it in edgeR).

**Median of ratios** (Anders & Huber 2010, DESeq/DESeq2) compares each sample with a pseudo-reference sample, the
geometric mean of every gene across samples:

$$s_i = \operatorname*{median}_{g\,:\,\text{all } y_{gj} > 0}\ \frac{y_{gi}}{\big(\prod_{j=1}^{n} y_{gj}\big)^{1/n}} .$$

*In words:* for each gene, ask "how many times more reads does sample $i$ have than a typical sample?"; then take the
**median** over genes. If most genes do not change, the median ratio reflects depth and composition — and the few
genes that change a lot do not move a median.

In [ ]:
# ── 4.1 Median-of-ratios size factors ──
def median_of_ratios(Y):
    """DESeq-style size factors: median over genes (all counts > 0) of count / geometric mean across samples."""
    Y = np.asarray(Y, float)
    ok = (Y > 0).all(axis=1)
    log_geo = np.log(Y[ok]).mean(axis=1, keepdims=True)
    return np.exp(np.median(np.log(Y[ok]) - log_geo, axis=0))


def total_count_factors(Y):
    """Library-size scaling, rescaled to geometric mean 1 (comparable with median of ratios)."""
    lib = np.asarray(Y, float).sum(axis=0)
    return lib / np.exp(np.log(lib).mean())


sf_table = pd.DataFrame({"true": s_true, "median of ratios": median_of_ratios(Y), "total count": total_count_factors(Y)},
                        index=counts.columns)
print("🧪 size factors for the main experiment (geometric mean 1):")
print(sf_table.round(3).T.to_string())

🔍 **What just happened?** In the main experiment both estimates are close to the truth: the 200 changed genes go up
and down in roughly equal numbers, so the composition hardly moves. Now let us provoke the failure: a second SIMULATED
experiment in which 20 abundant genes are induced 20-fold by the treatment (think of a strongly secreted protein or an
immunoglobulin in activated B cells).

In [ ]:
# ── 4.2 🧪 Provoking composition bias ──
comp = simulate_bulk_counts(np.random.default_rng(SEED + 6), n_dominant=20)
Yc, tc = comp["counts"].values, comp["samples"].treated.values
dominant = comp["genes"].true_log2fc.values > 4
print("share of reads taken by the 20 induced genes:", (Yc[dominant].sum(0) / Yc.sum(0)).round(2).tolist())
Xc = np.column_stack([np.ones(len(tc)), tc])
nullc = ~comp["genes"].is_de.values

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
rows = []
for ax, (name, sf) in zip(axes, [("total count", total_count_factors(Yc)), ("median of ratios", median_of_ratios(Yc))]):
    bc, _, _, _ = glm_irls(Yc, Xc, np.log(sf))
    l2 = bc[:, 1] / np.log(2)
    mean_expr = np.log10((Yc / sf).mean(1) + 1)
    ax.scatter(mean_expr[nullc], l2[nullc], s=3, color="0.55", alpha=0.5, label="truly unchanged")
    ax.scatter(mean_expr[~nullc], l2[~nullc], s=5, color=CLASS_COLORS[4], label="truly changed")
    ax.axhline(0, color="k", lw=0.8); ax.axhline(np.median(l2[nullc]), color=CLASS_COLORS[1], ls="--", lw=1.5,
                                                    label="median of unchanged genes")
    ax.set_title(f"size factors: {name}"); ax.set_xlabel("log10 mean normalised count"); ax.set_ylim(-4.5, 5.5)
    rows.append({"size factors": name, "max |error| vs true factors": np.abs(np.log(sf / comp['samples'].true_size_factor.values)).max(),
                 "median log2FC of unchanged genes": np.median(l2[nullc])})
axes[0].set_ylabel("estimated log2 fold change"); axes[1].legend(loc="lower right", fontsize=7)
plt.suptitle("SIMULATED composition bias: MA plots of the same counts", y=1.02)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(3).to_string(index=False))

🔍 **What just happened?** In the treated samples the 20 induced genes swallow about 60 % of all reads (7–8 % in the
controls). With total-count size factors, the cloud of 1,780 *unchanged* genes is shifted downwards by about 1.3 log2
units (left MA plot — "M" for log ratio, "A" for average): a biologist would report a "global down-regulation" to less
than half that never happened. The median of ratios ignores the few extreme genes and puts the unchanged genes back on
zero (right; median log2FC −0.01), and its size factors are within 2 % of the true ones (largest log error 0.02, against
0.50 for total counts).

⚠️ **The assumption behind every normalisation.** Median of ratios (and TMM) assumes that *most genes do not change*, or
change symmetrically. If a treatment really changes total mRNA per cell (e.g. a global transcriptional amplifier, or
cells that grow in size), no read-count-based method can tell — you need external **spike-in** RNAs added in equal
amounts per cell (ERCC controls) or cell counts.

✏️ **Your turn.** Set `n_dominant=60` in cell 4.2. How far do the unchanged genes move with total-count scaling? Does the
median of ratios still hold?

---
## 5 · Testing many genes

> 📍 **Where we are.** We have fold-change estimates and standard errors for every gene. Now we turn them into decisions
> for 2,000 genes at once — and measure, against the truth, how many decisions are wrong. From here on we use estimated
> (median-of-ratios) size factors, as a real analysis must.

### Two tests for one coefficient

The **Wald test** divides the estimate by its standard error:

$$z = \frac{\hat\beta_1}{\operatorname{SE}(\hat\beta_1)}, \qquad p = 2\,\Phi(-|z|), \qquad \operatorname{SE}(\hat\beta_1) = \sqrt{\big[(X^\top WX)^{-1}\big]_{11}} .$$

*In words:* how many standard errors is the log fold change away from zero? $\Phi$ is the standard normal cumulative
distribution; $p$ is the probability of a $|z|$ at least this large if the gene did not change.

The **likelihood-ratio test (LRT)** compares the best fit *with* the treatment coefficient to the best fit *without* it:

$$\Lambda = 2\big[\ell(\hat{\boldsymbol\beta}_{\text{full}}) - \ell(\hat{\boldsymbol\beta}_{\text{reduced}})\big] \ \sim\ \chi^2_1 \text{ under } H_0 .$$

*In words:* how much does the log-likelihood improve when the gene is allowed to change? Wilks' theorem says that, for
large samples, twice the improvement follows a chi-squared distribution with as many degrees of freedom as parameters
dropped (here one). The LRT also tests *several* coefficients at once (a factor with three levels, Bonus).

In [ ]:
# ── 5.1 Wald and likelihood-ratio tests (dispersions fixed at the oracle values for now) ──
sf_est = median_of_ratios(Y)
off_est = np.log(sf_est)
b_full, se_full, mu_full, _ = glm_irls(Y, X, off_est, true_alpha)
_, _, mu_red, _ = glm_irls(Y, X[:, :1], off_est, true_alpha)            # reduced model: intercept only
p_wald = wald_pvalues(b_full, se_full)
lrt = 2 * (nb_loglik(Y, mu_full, true_alpha) - nb_loglik(Y, mu_red, true_alpha))
p_lrt = stats.chi2.sf(lrt, df=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].loglog(p_wald, p_lrt, ".", ms=3, color="0.4"); axes[0].plot([1e-30, 1], [1e-30, 1], color=CLASS_COLORS[1], lw=1)
axes[0].set_xlim(1e-30, 1.5); axes[0].set_ylim(1e-30, 1.5)
axes[0].set_xlabel("Wald p-value"); axes[0].set_ylabel("LRT p-value"); axes[0].set_title("Wald and LRT agree (SIMULATED)")
axes[1].hist([p_wald[null], p_wald[~null]], bins=20, range=(0, 1), stacked=True, color=["0.7", CLASS_COLORS[4]],
             label=["truly unchanged", "truly changed"], edgecolor="w")
axes[1].set_xlabel("Wald p-value"); axes[1].set_ylabel("number of genes"); axes[1].legend()
axes[1].set_title("p-value histogram: flat null + spike near 0")
plt.tight_layout(); plt.show()
print(f"Spearman correlation Wald vs LRT: {stats.spearmanr(p_wald, p_lrt)[0]:.4f}")
print(f"p < 0.05: {np.sum(p_wald < 0.05)} genes, of which {np.sum((p_wald < 0.05) & null)} are truly unchanged")

🔍 **What just happened?** The two tests give almost identical p-values (they are asymptotically equivalent; they
differ most for genes with very small counts). The p-value histogram has the textbook shape — flat for the unchanged
genes plus a spike near zero from the changed ones — exactly the shape you saw for GWAS p-values in Module 06. But the
last line shows the problem of scale: "$p < 0.05$" selected 279 genes, 103 of them truly unchanged, because 5 % of 1,800
null genes is already about 90. We need a multiple-testing rule.

### FWER versus FDR

> 🔗 **Builds on** [Module 06](06_linear_regression_gwas.ipynb), Section 5.3: the **Bonferroni** correction rejects only if
> $p < \alpha/m$ for $m$ tests and controls the **family-wise error rate** (FWER) — the probability of even *one* false
> positive. In a GWAS of a million SNPs that is the right goal; in an expression screen it is usually too strict.

In a screen we accept a few false leads if most of what we follow up is real. The **false discovery rate** is the
expected *fraction* of false discoveries among all discoveries:

$$\text{FDR} = \mathbb E\left[\frac{V}{\max(R, 1)}\right], \qquad V = \#\text{ false discoveries}, \quad R = \#\text{ discoveries} .$$

*In words:* "of the genes I call changed, what fraction do I expect to be wrong?"

**The Benjamini–Hochberg (BH) step-up rule** (1995): sort the $m$ p-values, $p_{(1)} \le \dots \le p_{(m)}$; find the
**largest** rank $k$ with $p_{(k)} \le \frac{k}{m}q$; reject the $k$ hypotheses with the smallest p-values. This controls
the FDR at level $q$ when the tests are independent (and under many kinds of positive dependence). The **adjusted
p-value** ("q-value" in many tools) of each gene is the smallest $q$ at which it would be rejected; the toolkit's
`benjamini_hochberg` returns it.

In [ ]:
# ── 5.2 Benjamini–Hochberg by hand on six p-values ──
p6 = np.array([0.001, 0.008, 0.039, 0.040, 0.041, 0.60])
m, q_level = len(p6), 0.05
bh = pd.DataFrame({"rank k": np.arange(1, m + 1), "p_(k)": np.sort(p6), "threshold k/m·q": np.arange(1, m + 1) / m * q_level})
bh["p ≤ threshold?"] = bh["p_(k)"] <= bh["threshold k/m·q"]
k_star = bh.loc[bh["p ≤ threshold?"], "rank k"].max()
bh["rejected (step-up)"] = bh["rank k"] <= k_star
bh["adjusted p (toolkit)"] = benjamini_hochberg(np.sort(p6))
print(bh.to_string(index=False))
print(f"\nlargest k with p_(k) ≤ k·q/m: k* = {k_star} → reject {k_star} hypotheses;  Bonferroni (p < {q_level / m:.4f}) rejects {np.sum(p6 < q_level / m)}")

🔍 **What just happened?** Ranks 3 and 4 ($p = 0.039 > 0.025$ and $0.040 > 0.033$) fail their own thresholds, yet they
are rejected — because rank 5 ($p = 0.041 \le 0.0417$) passes, and BH rejects *everything up to the largest passing
rank*. That is the "step-up" part: a run of similar small p-values supports each other. The adjusted p-values tell the
same story: the first five are all $\le 0.05$ (0.0492). Bonferroni rejects only two. Now the real thing: 2,000 genes, graded against the truth.

In [ ]:
# ── 5.3 🧪 Realised FDR and power: Bonferroni vs BH, oracle vs estimated dispersions ──
def grade(pvalues, is_null, level=0.05, method="BH"):
    """Discoveries, false discoveries, realised FDR and power against the truth (untested genes: p = NaN)."""
    p = np.asarray(pvalues, float)
    tested = ~np.isnan(p)
    reject = np.zeros(len(p), bool)
    if method == "BH":
        reject[tested] = benjamini_hochberg(p[tested]) <= level
    else:
        reject[tested] = p[tested] <= level / tested.sum()
    false = int(np.sum(reject & is_null))
    return {"discoveries": int(reject.sum()), "false": false, "realised FDR": float(false / max(reject.sum(), 1)),
            "power": float(np.sum(reject & ~is_null) / np.sum(~is_null))}


alpha_hat = np.full(len(Y), 0.1)                       # gene-wise ML dispersions with estimated size factors
for rnd in range(3):
    b_tmp, _, mu_tmp, _ = glm_irls(Y, X, off_est, alpha_hat)
    alpha_hat = fit_dispersion_ml(Y, mu_tmp)
b_ml, se_ml, _, _ = glm_irls(Y, X, off_est, alpha_hat)
p_ml = wald_pvalues(b_ml, se_ml)

rows = {"oracle dispersions, Bonferroni": grade(p_wald, null, method="Bonferroni"),
        "oracle dispersions, BH q<0.05": grade(p_wald, null),
        "gene-wise ML dispersions, BH q<0.05": grade(p_ml, null),
        "Poisson GLM, BH q<0.05": grade(wald_pvalues(*glm_irls(Y, X, off_est)[:2]), null)}
print("🧪 2,000 SIMULATED genes, 200 truly changed (4 vs 4 samples):")
print(pd.DataFrame(rows).T.round(3).to_string())

In [ ]:
# ── 5.4 Volcano plots coloured by the hidden truth ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, (name, b, p) in zip(axes, [("oracle dispersions", b_full, p_wald), ("gene-wise ML dispersions", b_ml, p_ml)]):
    q = benjamini_hochberg(p)
    l2 = b[:, 1] / np.log(2)
    nlp = -np.log10(np.maximum(p, 1e-30))
    ax.scatter(l2[null], nlp[null], s=4, color="0.6", alpha=0.6, label="truly unchanged")
    ax.scatter(l2[~null], nlp[~null], s=6, color=CLASS_COLORS[4], label="truly changed")
    fp = (q <= 0.05) & null
    ax.scatter(l2[fp], nlp[fp], s=30, facecolor="none", edgecolor=CLASS_COLORS[1], lw=1, label="false discovery")
    if np.any(q <= 0.05):
        ax.axhline(-np.log10(p[q <= 0.05].max()), color="k", ls="--", lw=1, label="BH q = 0.05")
    ax.set_xlabel("estimated log2 fold change"); ax.set_title(f"{name}: {fp.sum()} false discoveries")
axes[0].set_ylabel("−log10 p-value"); axes[0].legend(fontsize=7)
plt.suptitle("SIMULATED volcano plots (4 vs 4 samples)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the 🧪 table row by row.

- **Bonferroni** (oracle dispersions) makes no false discovery but finds the fewest genes (115, power 0.58) — it
  protects against even one error.
- **BH with the oracle dispersions** finds 154 genes (power 0.75) with 5 false discoveries, a realised FDR of 3.2 % —
  within the promised 5 %: BH works as advertised *when the p-values are valid*.
- **BH with gene-wise ML dispersions** reports 269 discoveries, but 105 of them are false — a realised FDR of 39 %,
  nearly eight times the nominal level. BH did nothing wrong: it was fed p-values that are too small (Section 3.4), and
  "garbage in, garbage out" applies to FDR control too. In the volcano plot the false discoveries (red circles) are
  genes with modest fold changes but tiny p-values — genes whose dispersion was underestimated.
- **Poisson** is worse still: 908 discoveries, 79 % of them false.

The power column also reminds us that even a perfect method cannot find every changed gene: changes of 0.4–0.7 log2
units in weakly expressed genes are invisible with four replicates.

✏️ **Your turn.** Use `grade(p_wald, null, level=0.10)` and `level=0.01`. How do the realised FDR and power trade off?
Then compute the power separately for genes with `truth.base_mean < 50` and `>= 50` (hint: pass `p_wald[mask]`,
`null[mask]`). Why is the power so different?

### 📄 Research connection — False discoveries at genome scale

- **What the field does.** Genomics made multiple testing an everyday problem. Benjamini & Hochberg (1995) defined the false discovery
    rate and the step-up procedure; Storey (2002) introduced the q-value and the estimation of the proportion of true
    nulls π₀ (which makes BH less conservative when many genes change), and Storey & Tibshirani (2003) brought these ideas
    to genome-wide studies. Bourgon, Gentleman & Huber (2010) showed that
    **independent filtering** — removing genes with very low counts *before* testing, using a criterion that is
    independent of the test statistic under the null — increases power; DESeq2 does it automatically.
- **What this notebook builds.** We implemented the BH rule by hand, used the toolkit's adjusted p-values, and — something real data never
    allow — measured the realised FDR and power against a known truth. The key finding is that FDR control is only as good
    as the p-values: valid with the true dispersions, badly violated with noisy gene-wise estimates.
- **Where the frontier is.** Current directions include FDR control that uses covariates to weight hypotheses (e.g. independent
    hypothesis weighting, which up-weights well-expressed genes), knockoff-based FDR control, and the open problem of
    selective inference after data-driven steps such as clustering — testing for differences between clusters that were
    found in the same data inflates false discoveries (Module 30 returns to this).
- **Read next.** Benjamini & Hochberg 1995 (*JRSS-B*), *Controlling the false discovery rate*; Storey & Tibshirani 2003 (*PNAS*), *Statistical significance for genomewide studies*.

---
## 6 · Few replicates → shrink the dispersion (empirical Bayes)

> 📍 **Where we are.** Gene-wise dispersions estimated from 4 + 4 samples are biased and noisy, and the resulting
> p-values break FDR control. The fix used by DESeq2 and edgeR has two parts: a bias correction, and **borrowing
> strength across genes**.

### Part 1 — a bias correction (Cox–Reid)

The ML dispersion is biased low for the same reason the ML variance divides by $N$: it treats the fitted means as if
they were known. Cox & Reid's **adjusted profile likelihood** subtracts a penalty for the coefficients that were
estimated:

$$\ell_{\text{CR}}(\alpha) = \ell(\alpha;\, \hat{\boldsymbol\mu}) - \tfrac12 \log\det\big(X^\top W(\alpha) X\big), \qquad W(\alpha) = \operatorname{diag}\!\Big(\frac{\hat\mu_i}{1 + \alpha\hat\mu_i}\Big) .$$

*In words:* the more precisely the coefficients are pinned down for a given $\alpha$, the more the fit has "used up"
the data, and the more the likelihood is discounted. edgeR and DESeq2 both maximise this adjusted likelihood.

### Part 2 — share information across genes

Genes with similar expression levels have similar dispersions (Section 1.3: the parabola). So we (i) fit a smooth
**trend** $\alpha_{\text{tr}}(\bar\mu) = a_0 + a_1/\bar\mu$ through all gene-wise estimates, (ii) treat it as a **prior**,
$\log\alpha_g \sim \mathcal N\big(\log\alpha_{\text{tr}}(\bar\mu_g),\ \sigma^2_{\text{prior}}\big)$, and (iii) report each gene's
**posterior mode** (MAP). For intuition, if the gene's own log-estimate had a Gaussian sampling error with variance
$\sigma^2_{\text{gene}}$, the posterior mean would be the precision-weighted average

$$\log\tilde\alpha_g = \frac{\log\hat\alpha_g/\sigma^2_{\text{gene}} + \log\alpha_{\text{tr}}/\sigma^2_{\text{prior}}}{1/\sigma^2_{\text{gene}} + 1/\sigma^2_{\text{prior}}} .$$

*In words:* a compromise between what this gene's few samples say and what thousands of similar genes say, each
weighted by how much we trust it. With few replicates $\sigma^2_{\text{gene}}$ is large and the trend dominates; with
many replicates the gene speaks for itself.

> 🔗 **Builds on** [Module 21 · Bayesian learning](21_bayesian_learning_mcmc.ipynb): the posterior mean of a Gaussian
> mean with a Gaussian prior is exactly this precision-weighted average, and [Module 05](05_likelihood_mle_map.ipynb)
> showed that MAP shrinks towards the prior mean like ridge regression. The new twist is **empirical Bayes**: the prior
> (trend and $\sigma^2_{\text{prior}}$) is *estimated from the data of all genes* instead of being chosen by hand.

DESeq2 estimates $\sigma^2_{\text{prior}}$ as the spread of $\log\hat\alpha_g$ around the trend *minus* the spread
expected from sampling error alone, $\psi_1\big((n-p)/2\big)$ ($\psi_1$ is the trigamma function, the variance of the
log of a scaled $\chi^2$ variable), with a floor of 0.25. We maximise the actual NB likelihood times the prior (MAP),
which behaves sensibly even for genes whose estimate is pinned at zero.

In [ ]:
# ── 6.1 Cox–Reid adjusted dispersions ──
def cr_loglik(Y, mu, alpha, X):
    """Cox–Reid adjusted profile log-likelihood of the dispersion, one value per gene."""
    W = mu / (1 + np.asarray(alpha)[:, None] * mu)
    return nb_loglik(Y, mu, alpha) - 0.5 * np.linalg.slogdet(np.einsum("np,gn,nq->gpq", X, W, X))[1]


def fit_dispersion_cr(Y, X, offset, n_rounds=3):
    """Alternate IRLS for beta and golden-section search for the CR-adjusted dispersion of every gene."""
    alpha = np.full(len(Y), 0.1)
    for _ in range(n_rounds):
        _, _, mu, _ = glm_irls(Y, X, offset, alpha)
        alpha = np.exp(golden_max(lambda la: cr_loglik(Y, mu, np.exp(la), X), np.log(1e-8), np.log(10.0), len(Y)))
    return alpha


alpha_cr = fit_dispersion_cr(Y, X, off_est)
for name, a in [("gene-wise ML", alpha_hat), ("gene-wise Cox–Reid", alpha_cr)]:
    print(f"{name:20s}: median estimate/truth = {np.median(a / true_alpha):.2f};  pinned at the lower bound: {np.mean(a < 1e-6):.1%}")

In [ ]:
# ── 6.2 The mean–dispersion trend: a gamma-family GLM (a GLM again!) ──
def fit_dispersion_trend(alpha, mean, n_iter=10):
    """Fit alpha ≈ a0 + a1/mean by IRLS for a gamma GLM with identity link (weights 1/fitted², as in DESeq2)."""
    use = (alpha > 1e-6) & (mean > 0)                   # as DESeq2: skip genes whose estimate is pinned at zero
    Xt = np.column_stack([np.ones(use.sum()), 1 / mean[use]])
    coef = np.array([0.05, 1.0])
    for _ in range(n_iter):
        w = 1 / (Xt @ coef) ** 2                         # gamma variance ∝ mean², so weight by 1/fitted²
        coef = np.linalg.solve(Xt.T @ (w[:, None] * Xt), Xt.T @ (w * alpha[use]))
        coef = np.maximum(coef, 1e-6)                    # keep the trend positive
    return coef


base_mean = (Y / sf_est).mean(axis=1)
coef = fit_dispersion_trend(alpha_cr, base_mean)
alpha_trend = coef[0] + coef[1] / base_mean
print(f"fitted trend: alpha_tr = {coef[0]:.4f} + {coef[1]:.3f}/mean   (simulator: 0.0300 + 1.000/mean, times gene scatter)")

grid = np.logspace(0, 4.5, 100)
plt.figure(figsize=(7, 3.6))
plt.scatter(base_mean, np.maximum(alpha_cr, 1e-4), s=4, color="0.6", alpha=0.6, label="gene-wise Cox–Reid estimates")
plt.scatter(base_mean, true_alpha, s=2, color=CLASS_COLORS[0], alpha=0.3, label="true dispersions (hidden)")
plt.plot(grid, coef[0] + coef[1] / grid, color=CLASS_COLORS[4], lw=2.5, label="fitted trend (prior mean)")
plt.xscale("log"); plt.yscale("log"); plt.ylim(1e-4, 5)
plt.xlabel("mean normalised count"); plt.ylabel(r"dispersion $\alpha$ (estimates floored at 1e-4)")
plt.title("SIMULATED: noisy gene-wise estimates scatter around a smooth trend"); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The Cox–Reid correction removed most of the downward bias (the median ratio to the truth
moved from 0.57 to 0.85) and halved the share of genes stuck at zero (10 % → 5 %). The gene-wise estimates (grey)
scatter over more than an order of magnitude around the true values (blue), but their *average* behaviour as a function
of the mean is captured by a two-parameter curve. Its asymptote $a_0 = 0.033$ is close to the simulator's 0.03; the
low-count coefficient $a_1 = 1.46$ is larger than the simulator's 1.0, because the genes whose estimate was pinned at
zero — mostly weakly expressed ones — are left out of the fit (as in DESeq2), which lifts the low-count end of the curve.
That errs on the safe (conservative) side. The trend fit is itself a GLM: a gamma distribution (variance ∝ mean²) with an
identity link, fitted by IRLS — the same algorithm as Section 2 with different weights. Now we shrink.

In [ ]:
# ── 6.3 Empirical-Bayes MAP dispersions ──
def shrink_dispersions(Y, X, offset, alpha_gene, alpha_trend):
    """MAP of log alpha under a normal prior centred on the trend; prior variance estimated from all genes."""
    n, p = X.shape
    ok = alpha_gene > 1e-6                                       # genes not pinned at the lower bound
    resid = np.log(alpha_gene[ok]) - np.log(alpha_trend[ok])
    var_obs = stats.median_abs_deviation(resid, scale="normal") ** 2   # robust spread around the trend
    var_sampling = special.polygamma(1, (n - p) / 2)              # expected from sampling error alone
    var_prior = max(var_obs - var_sampling, 0.25)
    _, _, mu, _ = glm_irls(Y, X, offset, alpha_trend)
    log_post = lambda la: cr_loglik(Y, mu, np.exp(la), X) - (la - np.log(alpha_trend)) ** 2 / (2 * var_prior)
    alpha_map = np.exp(golden_max(log_post, np.log(1e-8), np.log(10.0), len(Y)))
    return alpha_map, dict(var_obs=var_obs, var_sampling=var_sampling, var_prior=var_prior)


alpha_map, info = shrink_dispersions(Y, X, off_est, alpha_cr, alpha_trend)
print({k: round(float(v), 3) for k, v in info.items()})

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
for ax, (name, a) in zip(axes, [("gene-wise Cox–Reid", alpha_cr), ("empirical-Bayes MAP (shrunken)", alpha_map)]):
    ax.scatter(true_alpha, np.maximum(a, 1e-4), s=4, alpha=0.5, color="0.4")
    ax.plot([1e-3, 3], [1e-3, 3], color=CLASS_COLORS[1], lw=1.5, label="estimate = truth")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_ylim(1e-4, 5)
    ax.set_xlabel("true dispersion (hidden)"); ax.set_title(name)
axes[0].set_ylabel("estimated dispersion"); axes[0].legend()
plt.suptitle("SIMULATED: shrinkage pulls wild estimates towards plausible values", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.4 🧪 Which dispersion estimate gives honest tests? ──
def common_dispersion(Y, X, offset):
    """One dispersion shared by all genes (maximises the summed CR likelihood) — 'complete pooling'."""
    _, _, mu, _ = glm_irls(Y, X, offset, 0.05)
    total = lambda la: np.array([cr_loglik(Y, mu, np.full(len(Y), np.exp(la[0])), X).sum()])
    return float(np.exp(golden_max(total, np.log(1e-4), np.log(1.0), 1)[0]))


candidates = {"gene-wise ML (no pooling)": alpha_hat, "gene-wise Cox–Reid": alpha_cr,
              "common dispersion (complete pooling)": np.full(len(Y), common_dispersion(Y, X, off_est)),
              "trend only": alpha_trend, "empirical-Bayes MAP (partial pooling)": alpha_map, "truth (oracle)": true_alpha}
rows = {}
for name, a in candidates.items():
    b, se, _, _ = glm_irls(Y, X, off_est, a)
    err = np.log(np.maximum(a, 1e-8)) - np.log(true_alpha)
    rows[name] = {"median |log error|": np.median(np.abs(err)), "null p<0.05": np.mean(wald_pvalues(b, se)[null] < 0.05),
                  **grade(wald_pvalues(b, se), null)}
print("🧪 4 vs 4 SIMULATED samples, BH q < 0.05:")
print(pd.DataFrame(rows).T.round(3).to_string())

🔍 **What just happened?** First, the prior: the gene-wise log-estimates scatter around the trend with variance 0.82,
of which 0.40 is expected from sampling error alone, leaving an estimated prior variance of 0.43 (the simulator used
$0.5^2 = 0.25$). The shrunken estimates (right plot) hug the diagonal far more tightly than the gene-wise ones (left).
The table is the heart of DESeq2/edgeR, measured against the truth:

- **No pooling** (gene-wise ML) has the largest dispersion errors and the worst FDR (39 %); Cox–Reid helps but not
  enough (29 %) — four replicates per group simply do not pin down a variance.
- **Complete pooling** (one dispersion for all genes) ignores the strong dependence on the mean (FDR 12 %).
- **Trend only** is much better (FDR 6 %), but treats genes that are genuinely more variable than the trend as ordinary.
- **Partial pooling** (empirical-Bayes MAP) has the smallest dispersion error of all estimates (median |log error| 0.28)
  and brings the realised FDR from 39 % down to 8.5 %, with the same power as the oracle (0.75).

In this particular data set the trend-only fit happens to make four fewer false discoveries than the MAP (10 vs 14);
both are still somewhat liberal compared with the oracle (5 false discoveries). One reason is that genes whose true
dispersion lies far above the trend are shrunk too much. DESeq2 handles such "dispersion outliers" by keeping their
gene-wise estimate when it lies more than two standard deviations of the log-dispersion residuals above the trend, and
edgeR's quasi-likelihood
F-tests account for the uncertainty of the dispersion itself. A single simulated data set also gives a *noisy* realised
FDR — with 150–170 discoveries, one false discovery more or less changes it by more than half a percentage point — so
the next cell repeats the comparison on fresh experiments.

DESeq2 also shrinks the **fold changes** (Love *et al.* 2014; newer versions offer the heavier-tailed apeglm prior): a
huge log fold change estimated from a handful of reads is pulled towards zero, which makes gene rankings more stable.
The idea is the same partial pooling, applied to $\beta_1$ instead of $\alpha$.

Let us package everything — size factors, Cox–Reid dispersions, trend, MAP, Wald test — into one function, our
"mini-DESeq2", and test it on new SIMULATED experiments with 3, 4 and 6 replicates per group.

In [ ]:
# ── 6.5 A mini-DESeq2 and a replication study ──
def nb_glm_de(Y, X, coef=1, offset=None):
    """NB GLM differential expression: median-of-ratios offsets (unless given), Cox–Reid dispersions shrunk to a
    mean trend (empirical Bayes), Wald test of column `coef` of X. Returns a per-gene DataFrame."""
    Y = np.asarray(Y, float)
    offset = np.log(median_of_ratios(Y)) if offset is None else offset
    out = pd.DataFrame(np.nan, index=range(len(Y)), columns=["base_mean", "log2FC", "lfcSE", "pvalue", "padj",
                                                             "dispersion", "dispersion_gene"])
    t = Y.sum(axis=1) > 0                                   # all-zero genes cannot be tested (DESeq2 reports NA)
    Yt = Y[t]
    a_gene = fit_dispersion_cr(Yt, X, offset)
    mean = (Yt / np.exp(offset)).mean(axis=1)
    c = fit_dispersion_trend(a_gene, mean)
    a_trend = c[0] + c[1] / mean
    a_map, _ = shrink_dispersions(Yt, X, offset, a_gene, a_trend)
    b, se, _, _ = glm_irls(Yt, X, offset, a_map)
    p = wald_pvalues(b, se, coef)
    out.loc[t, :] = np.column_stack([mean, b[:, coef] / np.log(2), se[:, coef] / np.log(2), p, benjamini_hochberg(p),
                                     a_map, a_gene])
    return out


rows = []
t0 = time.time()
for npg in [3, 4, 6]:
    for rep in range(3):                                   # three fresh experiments per design
        sim = simulate_bulk_counts(np.random.default_rng(SEED + 100 * npg + rep), n_per_group=npg)
        Ys, ts = sim["counts"].values, sim["samples"].treated.values
        Xs = np.column_stack([np.ones(len(ts)), ts])
        res = nb_glm_de(Ys, Xs)
        b_g, se_g, _, _ = glm_irls(Ys, Xs, np.log(median_of_ratios(Ys)), res.dispersion_gene.fillna(0.1).values)
        p_gene = np.where(res.pvalue.isna(), np.nan, wald_pvalues(b_g, se_g))
        nul = ~sim["genes"].is_de.values
        g_gene, g_shr = grade(p_gene, nul), grade(res.pvalue.values, nul)
        rows.append({"design": f"{npg} vs {npg}", "FDR gene-wise CR": g_gene["realised FDR"],
                     "FDR shrunken": g_shr["realised FDR"], "power gene-wise CR": g_gene["power"],
                     "power shrunken": g_shr["power"]})
summary = pd.DataFrame(rows).groupby("design").agg(["mean", "min", "max"]).round(3)
print(f"🧪 9 fresh SIMULATED experiments, BH q < 0.05 ({time.time() - t0:.1f} s) — mean, min and max over 3 per design:")
print(summary.to_string())

🔍 **What just happened?** On nine fresh experiments the pattern is consistent: with gene-wise (Cox–Reid) dispersions
the realised FDR averages 33 % with three replicates per group, 22 % with four and 16 % with six; with the shrunken
dispersions it averages 8.7 %, 6.5 % and 6.0 % — close to, though on average slightly above, the nominal 5 %. Shrinkage
costs a little power (e.g. 0.69 vs 0.74 with three replicates), because the gene-wise version was "finding" genes
partly by underestimating their noise. More replicates raise the power of every method. This is the quantitative
reason why every modern RNA-seq tool shares information across genes, and why three replicates is a bare minimum rather
than a comfortable design. A real yeast experiment with dozens of biological replicates per condition (Schurch *et al.*
2016) reached a similar conclusion: with three replicates most tools found only a minority of the changed genes, and the
authors recommended at least six replicates per condition.

✏️ **Your turn.** Add `2` to the list `[3, 4, 6]` in cell 6.5 (two replicates per group). Does shrinkage still rescue
the FDR? What happens to the power? What does this tell you about experiments with "n = 2"?

### 📄 Research connection — Bulk RNA-seq differential expression with negative-binomial GLMs

- **What the field does.** After microarrays, RNA-seq brought counts. Robinson, McCarthy & Smyth (2010, **edgeR**) and Anders & Huber
    (2010, **DESeq**) modelled them with the negative binomial and shared information across genes to estimate
    dispersions from few replicates; **DESeq2** (Love, Huber & Anders 2014) added the empirical-Bayes MAP dispersions with
    a mean trend, fold-change shrinkage, outlier detection (Cook's distance) and independent filtering. **limma-voom**
    (Law *et al.* 2014) instead log-transforms counts and gives each observation a precision weight from the mean–variance
    trend, re-using limma's linear-model machinery; edgeR's quasi-likelihood F-tests account for the uncertainty in the
    dispersion. These three remain the most widely used tools for bulk data.
- **What this notebook builds.** Every core component of DESeq2, from scratch: size factors by median of ratios, the NB GLM fitted by IRLS
    with offsets, Cox–Reid adjusted dispersions, a gamma-GLM trend, empirical-Bayes MAP shrinkage and the Wald test with BH
    adjustment — and, unlike any real data set, a measurement of realised FDR and power. We omitted fold-change
    shrinkage, outlier handling and independent filtering, and our test is slightly liberal.
- **Where the frontier is.** Open questions include FDR control with very small samples and with large, heterogeneous human cohorts
    (where some benchmarks have reported that parametric methods can be anti-conservative and that rank-based tests
    become competitive), modelling unwanted variation that is not recorded in the design, isoform- and allele-specific
    expression, and the analysis of long-read RNA-seq counts.
- **Read next.** Love, Huber & Anders 2014 (*Genome Biology*), *Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2*; Law *et al.* 2014 (*Genome Biology*), *voom: precision weights unlock linear model analysis tools for RNA-seq read counts*.

---
## 7 · From bulk to single cells: pseudobulk

> 📍 **Where we are.** Single-cell RNA-seq ([Module 30](30_single_cell_rnaseq.ipynb)) measures thousands of cells per
> donor. It is tempting to compare *cells* between conditions — hundreds of "replicates"! This section shows, against
> the truth, why that is wrong and what to do instead.

We simulate a Kang-like experiment with `simulate_scrna`: six donors, the first three **control** and the last three
**interferon-stimulated**, six PBMC-like cell types. Each donor has its own small, gene-specific expression shifts
(`donor_effect`, a log-scale standard deviation of 0.15 by default) — real people differ. The truth `true_logfc` says
which genes respond in each cell type: 10 interferon genes in every type, plus 40 monocyte-specific responders.

The **unit of replication** is the thing that was independently assigned to a condition. Here it is the *donor*: the
cells of one donor share that donor's genome, history and sample handling. Treating 500 cells from three donors as 500
independent replicates is **pseudoreplication** — the same error as cross-validating biopsies instead of patients in
[Module 23](23_model_selection.ipynb).

In [ ]:
# ── 7.1 A Kang-like SIMULATED single-cell experiment ──
scd = simulate_scrna(np.random.default_rng(SEED + 7), n_cells=3000, n_genes=800, n_donors=6, n_batches=1)
obs, sc_counts = scd["obs"], scd["counts"]
print("SIMULATED scRNA-seq:", sc_counts.shape, "(cells × genes)")
print(pd.crosstab(obs.donor, obs.condition).to_string())

mono = (obs.true_type == "Monocyte").values
C_mono = sc_counts[mono].T.astype(float)                       # genes × cells, like the bulk matrix
stim_cell = (obs.condition[mono] == "stimulated").values.astype(float)
true_mono = scd["true_logfc"].loc["Monocyte"].values
expressed = C_mono.sum(axis=1) >= 10                           # drop genes with almost no reads in monocytes
null_sc = true_mono[expressed] == 0
print(f"\nmonocytes: {mono.sum()} cells; genes kept: {expressed.sum()}; "
      f"truly responding genes among them: {np.sum(~null_sc)} (hidden truth)")

In [ ]:
# ── 7.2 The tempting analysis: cells as replicates ──
Yc_m = C_mono[expressed]
L_mono = log_normalize(C_mono.T)[:, expressed]                  # cells × genes, log(1 + 10^4 · count/total)
p_wilcox = stats.mannwhitneyu(L_mono[stim_cell == 1], L_mono[stim_cell == 0], axis=0).pvalue

lib = C_mono.sum(axis=0)
X_cell = np.column_stack([np.ones_like(stim_cell), stim_cell])
b_c, se_c, _, _ = glm_irls(Yc_m, X_cell, np.log(lib / np.exp(np.log(lib).mean())))
p_cellpois = wald_pvalues(b_c, se_c)
print(f"Wilcoxon on {mono.sum()} cells and a cell-level Poisson GLM: done")

In [ ]:
# ── 7.3 Pseudobulk: sum the counts of each donor's monocytes, then the NB GLM of Section 6 ──
donors = sorted(obs.donor.unique())
mono_obs = obs[mono]
PB = np.column_stack([C_mono[:, (mono_obs.donor == d).values].sum(axis=1) for d in donors])   # genes × donors
stim_donor = np.array([(mono_obs.condition[mono_obs.donor == d] == "stimulated").all() for d in donors], float)
print("cells summed per donor:", [int((mono_obs.donor == d).sum()) for d in donors], "| stimulated:", stim_donor.astype(int).tolist())
X_pb = np.column_stack([np.ones(len(donors)), stim_donor])
pb = nb_glm_de(PB[expressed], X_pb)

rows = {"cell-level Wilcoxon (cells as replicates)": grade(p_wilcox, null_sc),
        "cell-level Poisson GLM (cells as replicates)": grade(p_cellpois, null_sc),
        "pseudobulk NB GLM (donors as replicates)": grade(pb.pvalue.values, null_sc)}
print("\n🧪 monocytes, stimulated vs control, BH q < 0.05:")
print(pd.DataFrame(rows).T.round(3).to_string())
print(f"pseudobulk: correlation of estimated and true log2FC for responding genes = "
      f"{np.corrcoef(pb.log2FC[~null_sc], true_mono[expressed][~null_sc] / np.log(2))[0, 1]:.3f}")
cpm = C_mono[expressed] / lib * 1e4                         # total-count-normalised cells (genes × cells)
m_stim, m_ctrl = cpm[:, stim_cell == 1].mean(axis=1), cpm[:, stim_cell == 0].mean(axis=1)
ok = null_sc & (m_stim > 0) & (m_ctrl > 0)
print(f"truly unchanged genes, median log(stim/ctrl): total-count-normalised cells {np.median(np.log(m_stim[ok] / m_ctrl[ok])):+.3f}, "
      f"pseudobulk with median of ratios {np.median(pb.log2FC.values[null_sc]) * np.log(2):+.3f}")

In [ ]:
# ── 7.4 Why: look at the worst false discovery of the cell-level test ──
g_fp = np.flatnonzero(null_sc)[np.argmin(p_wilcox[null_sc])]
expr_g = L_mono[:, g_fp]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].hist([p_wilcox[null_sc], pb.pvalue.values[null_sc]], bins=20, range=(0, 1), color=["0.6", CLASS_COLORS[0]],
             label=["cell-level Wilcoxon", "pseudobulk NB GLM"], edgecolor="w")
axes[0].set_xlabel("p-value (truly unchanged genes)"); axes[0].set_ylabel("number of genes"); axes[0].legend()
axes[0].set_title("null p-values: pile-up vs (roughly) flat")
jit = np.random.default_rng(SEED + 8)
for k, d in enumerate(donors):
    m = (mono_obs.donor == d).values
    col = CLASS_COLORS[1] if stim_donor[k] else CLASS_COLORS[0]
    axes[1].scatter(k + jit.uniform(-0.3, 0.3, m.sum()), expr_g[m], s=3, alpha=0.4, color=col)
    axes[1].plot([k - 0.35, k + 0.35], [expr_g[m].mean()] * 2, color="k", lw=2.5)
axes[1].set_xticks(range(len(donors))); axes[1].set_xticklabels([f"{d}\n({'stim' if s else 'ctrl'})" for d, s in zip(donors, stim_donor)], fontsize=8)
axes[1].set_ylabel("log-normalised expression")
axes[1].set_title(f"gene {scd['var'].index[expressed][g_fp]} (truly unchanged): Wilcoxon p = {p_wilcox[g_fp]:.1e}, "
                  f"pseudobulk p = {pb.pvalue.values[g_fp]:.2f}", fontsize=9)
plt.suptitle("SIMULATED pseudoreplication: donors differ, cells within a donor are not independent", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both cell-level analyses report impressive numbers of "significant" genes (91 and 139) — and
most of their discoveries are genes whose true stimulation effect is *zero* (realised FDRs of 58 % and 71 %). The
pseudobulk test, with only six samples, finds fewer genes (32, two thirds of the 48 true responders) but every one of them
is real in this run, and its fold changes track the truth closely (correlation 0.97).

The right panel shows the mechanism for the cell-level test's top false discovery. Each column is one donor; black bars
are donor means. The donors differ a little from each other — genuinely, as people do — and by chance the three
stimulated donors happen to sit on one side. A test on hundreds of cells "sees" hundreds of replicates and declares the
small difference overwhelming, but there are really only **three independent observations per condition**, and three
random donors differing a bit is not surprising at all. Pseudobulk summarises each donor by one number (a sum of
counts, which keeps the NB machinery), so the test's sample size is the number of donors — the honest one. Squair *et
al.* (2021), using matched bulk RNA-seq as a reference, showed that single-cell DE methods treating cells as replicates
are biased towards highly expressed genes and can report hundreds of differentially expressed genes even when there is
no biological difference, while pseudobulk methods performed best — the pattern our simulation reproduces.

⚠️ **More cells do not fix it.** Doubling the number of cells per donor makes the cell-level p-values *smaller* (more
false discoveries), not more honest. Only more donors increase the real information about donor-to-donor variability.

⚠️ **A second, smaller culprit: composition (Section 4 again).** Each cell has a fixed budget of UMIs, so when 50 genes
go up or down in stimulated monocytes, the other genes' *share* of the reads shifts slightly. The last line of cell 7.3
measures it: after total-count normalisation of each cell, the truly unchanged genes are about 4 % lower in stimulated
cells (median log ratio −0.045), whereas the median-of-ratios size factors of the pseudobulk absorb most of the shift
(median estimated log fold change −0.019). With hundreds of cells, cell-level tests on log-normalised data can detect even this
tiny shift for well-expressed genes, so part of their "false discoveries" is a real but biologically meaningless
compositional effect — one that real sequencing data have too.

✏️ **Your turn.** In cell 7.1 add `donor_effect=0.0` to the `simulate_scrna` call and re-run Section 7. How much more
honest do the cell-level tests become, and why not completely (hint: the composition note)? What does that tell you
about the main *source* of their false discoveries? Then try `donor_effect=0.3`.

### 📄 Research connection — Single-cell differential expression and pseudoreplication

- **What the field does.** Early single-cell DE compared cells with Wilcoxon tests or cell-level models (still the default "find
    markers" step of Seurat and scanpy, where it is fine for *ranking* marker genes of clusters). For comparisons between
    conditions with several individuals, Crowell *et al.* (2020, muscat) and Squair *et al.* (2021) showed that
    **pseudobulk** aggregation per sample and cell type, followed by edgeR/DESeq2/limma, controls false discoveries,
    whereas cell-level tests are dominated by between-individual variation; mixed models with a random effect per
    individual are the main alternative.
- **What this notebook builds.** We reproduced the core finding on a SIMULATED Kang-like design with known effects: cell-level Wilcoxon and
    Poisson tests have high realised FDR, the pseudobulk NB GLM controls it. The truth also lets us check fold changes and
    power, which real benchmarks can only approximate.
- **Where the frontier is.** Pseudobulk needs enough cells per sample and cell type and discards within-sample information; current
    work develops mixed models and cell-state-aware tests (e.g. differential expression along continuous states rather
    than discrete clusters), DE for rare cell types with few donors, and population-scale single-cell cohorts with
    hundreds of donors, which make genetic effects on cell-type-specific expression testable.
- **Read next.** Squair *et al.* 2021 (*Nature Communications*), *Confronting false discoveries in single-cell differential expression*; Crowell *et al.* 2020 (*Nature Communications*), *muscat detects subpopulation-specific state transitions from multi-sample multi-condition single-cell transcriptomics data*.

---
## 8 · Design matrices with covariates

> 📍 **Where we are.** Everything so far compared two groups. Because our model is a regression, any design matrix
> works: covariates for nuisance factors, interactions for "does the effect differ between …?". Two short examples.

**A batch covariate.** Suppose the eight bulk samples were processed on two days, two controls and two treated per day,
and each day shifts every gene a little. Adding a batch column to $X$,
$\log\mu_{gi} = \log s_i + \beta_0 + \beta_1\,\text{treated}_i + \beta_2\,\text{batch}_i$, removes the batch variation from the
residual noise. *In words:* compare treated and control *within* each day.

In [ ]:
# ── 8.1 Batch as a covariate (balanced design, SIMULATED) ──
bat = simulate_bulk_counts(np.random.default_rng(SEED + 9), batch_effect=0.5)
Yb, sb = bat["counts"].values, bat["samples"]
print(sb[["condition", "batch"]].T.to_string())
X_nobatch = np.column_stack([np.ones(8), sb.treated])
X_batch = np.column_stack([np.ones(8), sb.treated, sb.batch])
nul_b = ~bat["genes"].is_de.values
rows = {}
for name, Xd in [("treated only", X_nobatch), ("treated + batch", X_batch)]:
    r = nb_glm_de(Yb, Xd)
    rows[name] = {"median dispersion": r.dispersion.median(), **grade(r.pvalue.values, nul_b)}
print("\n🧪 BH q < 0.05:")
print(pd.DataFrame(rows).T.round(3).to_string())

🔍 **What just happened?** Without the batch column, the day-to-day shifts are absorbed into the dispersion (median
0.114 instead of 0.043), standard errors grow, and power drops to 0.57. With the batch column the dispersion returns to
its biological level and power rises to 0.74. Because the design is **balanced** (each day has both conditions), ignoring
batch costs power but does not bias the fold changes. In a **confounded** design — all controls on day 1, all treated on
day 2 — batch and treatment columns would be identical, $X^\top WX$ singular, and no statistical method could separate
them: the experiment would be uninterpretable ([Module 31](31_batch_integration.ipynb) returns to this). Note also the
realised FDR (8 %): with three coefficients estimated from eight samples, our approximate test is again somewhat liberal.

**An interaction.** Does stimulation change a gene *differently* in monocytes than in CD4 T cells? Take the pseudobulk
of both cell types for each donor (12 samples) and fit

$$\log\mu = \log s + \beta_0 + \beta_1\,\text{mono} + \beta_2\,\text{stim} + \beta_3\,(\text{mono}\times\text{stim}) .$$

*In words:* $\beta_2$ is the stimulation effect in CD4 T cells, $\beta_2 + \beta_3$ the effect in monocytes, and the
**interaction** $\beta_3$ is the *difference* between the two responses. Testing $\beta_3 = 0$ asks "is the response
cell-type-specific?". The simulation's truth: the 10 interferon genes respond equally in all types (true $\beta_3 = 0$),
the 40 monocyte-specific responders have $\beta_3 \neq 0$.

In [ ]:
# ── 8.2 A cell-type × stimulation interaction (pseudobulk of two cell types) ──
cols, meta = [], []
for ct in ["CD4 T", "Monocyte"]:
    for d in donors:
        m = ((obs.true_type == ct) & (obs.donor == d)).values
        cols.append(sc_counts[m].sum(axis=0)); meta.append((ct, d, obs.condition[m].iloc[0]))
PB2 = np.column_stack(cols)
meta = pd.DataFrame(meta, columns=["cell type", "donor", "condition"])
mono2, stim2 = (meta["cell type"] == "Monocyte").values.astype(float), (meta.condition == "stimulated").values.astype(float)
X_int = np.column_stack([np.ones(12), mono2, stim2, mono2 * stim2])
print(pd.concat([meta, pd.DataFrame(X_int.astype(int), columns=["intercept", "mono", "stim", "mono×stim"])], axis=1).to_string())

keep2 = PB2.sum(axis=1) >= 20
res_int = nb_glm_de(PB2[keep2], X_int, coef=3)
true_int = (scd["true_logfc"].loc["Monocyte"] - scd["true_logfc"].loc["CD4 T"]).values[keep2]
is_ifn = scd["var"].is_interferon.values[keep2]
print("\n🧪 interaction test (BH q < 0.05):", {k: round(v, 3) for k, v in grade(res_int.pvalue.values, true_int == 0).items()})
print(f"interferon genes (same response in both types) with interaction q < 0.05: {np.sum(res_int.padj.values[is_ifn] < 0.05)} of {is_ifn.sum()}")
res_stim = nb_glm_de(PB2[keep2], X_int, coef=2)
print(f"…but their stimulation effect in CD4 T cells (beta_2) has q < 0.05 for {np.sum(res_stim.padj.values[is_ifn] < 0.05)} of {is_ifn.sum()}")

🔍 **What just happened?** The design matrix makes the model explicit: one row per pseudobulk sample, one column per
coefficient. The interaction test finds about half of the monocyte-specific responders (21 genes, power 0.54) with no
false discoveries in this run, and correctly flags **none** of the 10 interferon genes — they respond strongly, but
*equally* in both cell types, as the separate test of $\beta_2$ confirms (9 of 10 significant in CD4 T cells). Asking "is the effect different?" requires testing the difference
directly: finding a gene significant in monocytes and not significant in T cells is **not** evidence of a
cell-type-specific response (the absence of evidence in T cells may just be low power) — a classic error in published
papers.

✏️ **Your turn.** Each donor contributes both a CD4 T and a monocyte sample, so the 12 samples are *paired*. Replace the
`stim` column by five donor indicator columns (donors 2–6; donor 1 is absorbed by the intercept) and refit the
interaction. Why does the stimulation main effect disappear from the model, while the interaction can still be tested?

### 📄 Research connection — GLMs as a universal tool in genomics

- **What the field does.** The count GLM appears far beyond bulk DE. **sctransform** (Hafemeister & Satija 2019) fits a regularised NB
    GLM of each gene on sequencing depth across cells and uses the Pearson residuals as normalised expression; Lause,
    Berens & Kobak (2021) showed that *analytic* Pearson residuals with a fixed overdispersion do as well, and Svensson
    (2020) that droplet UMI counts are well described by the NB without zero inflation. CRISPR screens count guide RNAs
    and test them with NB models (**MAGeCK**, Li *et al.* 2014); ChIP-seq, ATAC-seq and Hi-C differential analyses use the
    same machinery.
- **What this notebook builds.** A general IRLS engine for log-link count GLMs with offsets, arbitrary design matrices (covariates,
    interactions) and Wald/LRT tests — exactly what Module 30 re-uses for Pearson residuals and pseudobulk DE.
- **Where the frontier is.** Fast GLMs for millions of cells (glmGamPoi), GLM-based latent-variable models (GLM-PCA), and
    likelihood models inside deep generative models: scVI ([Module 27](27_variational_inference_vae.ipynb)) is, at its
    output, an NB GLM whose linear predictor is replaced by a neural network.
- **Read next.** Hafemeister & Satija 2019 (*Genome Biology*), *Normalization and variance stabilization of single-cell RNA-seq data using regularized negative binomial regression*; Lause, Berens & Kobak 2021 (*Genome Biology*), *Analytic Pearson residuals for normalization of single-cell RNA-seq UMI data*.

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran on simulations whose truth we knew. Here is what changes with real data —
> no downloads, just a map.

**Scale.** A real bulk study has ~20,000–60,000 genes and 6–100+ samples; a single-cell study has thousands to millions
of cells from a handful to hundreds of donors. Our vectorised IRLS scales linearly with the number of genes, so the
algorithms are the same; production tools add careful numerics (and, for millions of cells, specialised code such as
glmGamPoi).

**Public data to practise on.**
[GEO](https://www.ncbi.nlm.nih.gov/geo/) and [ArrayExpress](https://www.ebi.ac.uk/biostudies/arrayexpress) hold the raw
counts of most published experiments; [recount3](https://rna.recount.bio/) re-processed hundreds of thousands of
public human and mouse RNA-seq samples uniformly; [GTEx](https://gtexportal.org/) has bulk RNA-seq of ~50 tissues from
about a thousand donors; [TCGA](https://portal.gdc.cancer.gov/) has tumour RNA-seq with clinical data; the Kang *et al.*
(2018) interferon PBMC data and many other single-cell sets are available through
[CELLxGENE](https://cellxgene.cziscience.com/).

**Production tools.** [DESeq2](https://bioconductor.org/packages/DESeq2/), [edgeR](https://bioconductor.org/packages/edgeR/)
and [limma](https://bioconductor.org/packages/limma/) (voom) in R/Bioconductor; [PyDESeq2](https://github.com/owkin/PyDESeq2)
in Python; [glmGamPoi](https://bioconductor.org/packages/glmGamPoi/) for large single-cell GLMs; muscat for multi-sample
single-cell DE. The concepts map one-to-one onto this notebook: `sizeFactors` (4), `dispersions` and the trend (6),
`nbinomWaldTest`/`nbinomLRT` (5), `results(..., alpha=0.05)` with BH (5), `design = ~ batch + condition` (8).

**What goes wrong first.**

1. **Outliers and failed samples.** One degraded sample or one gene with a single extreme count can dominate. Look at
   PCA plots and sample correlations before testing; DESeq2 flags extreme counts with Cook's distance.
2. **Hidden confounders and batches.** Processing date, RNA quality, sex, cell-type composition of a tissue — if they
   are not in the design, they inflate dispersions (lost power) or, if correlated with the condition, create false
   discoveries. Record everything; methods such as surrogate variable analysis estimate unrecorded factors.
3. **Low counts and the multiple-testing burden.** Thousands of barely detected genes cannot reach significance but
   still count in the BH correction; independent filtering removes them. Zero counts in droplet single-cell data are
   mostly just low means, not a separate "dropout" process (Svensson 2020) — zero-inflated models are rarely needed for
   UMI data.

**What simulation cannot teach.** Real data have no answer key. The 🧪 checks of this notebook are replaced by
indirect evidence: p-value histograms (flat null + spike), replication in independent cohorts, spike-in controls,
and agreement between methods.

## 📌 Key takeaways

- **Sequencing produces counts.** Re-sequencing the same library gives Poisson noise (variance = mean); biological
  replicates add a term that grows with the square of the mean: $\operatorname{Var} = \mu + \alpha\mu^2$, the
  **negative binomial** (a Gamma–Poisson mixture).
- **A GLM** = random component + linear predictor $\eta = X\boldsymbol\beta$ + link. Logistic, Poisson and NB regression are
  siblings; all have gradient "features × (observed − expected)" and are fitted by Newton's method = **IRLS**. Library
  size enters as an **offset** $\log s_i$ (coefficient fixed at 1).
- **Size factors** must be robust to composition: the **median of ratios** is; total-count scaling makes unchanged genes
  look down-regulated when a few genes take over the read budget.
- **Wald and likelihood-ratio tests** turn coefficients into p-values; with thousands of genes, control the **FDR** with
  Benjamini–Hochberg (reject up to the largest $k$ with $p_{(k)} \le kq/m$) instead of the FWER (Bonferroni).
- **FDR control is only as good as the p-values.** Poisson p-values (and NB p-values with noisy gene-wise dispersions)
  are anti-conservative — in our 4 vs 4 simulation the realised FDR was several times the nominal 5 %.
- **Empirical Bayes rescues small experiments:** Cox–Reid-corrected dispersions shrunk towards a mean trend (partial
  pooling, the precision-weighted compromise of Module 21) brought the realised FDR close to the nominal level.
- **The donor, not the cell, is the unit of replication.** Cell-level tests in multi-donor single-cell data produce
  mostly false discoveries; **pseudobulk** + NB GLM controls them.
- **Design matrices** add covariates (batch) and interactions (cell-type-specific responses); test differences directly.

---
## ✅ Checkpoint

1. Why is library size entered as an offset rather than as an ordinary covariate?
   <details><summary>Answer</summary>Because we know its coefficient: doubling the sequencing depth doubles every expected count, so on the log scale $\log s_i$ enters with coefficient exactly 1. Estimating that coefficient from a handful of samples would waste a degree of freedom and add noise (and could absorb part of a treatment effect if depth differs between groups).</details>
2. Compute the BH decisions at q = 0.05 for the p-values 0.002, 0.009, 0.020, 0.040, 0.30, 0.80.
   <details><summary>Answer</summary>m = 6; thresholds k·0.05/6 = 0.0083, 0.0167, 0.025, 0.033, 0.042, 0.05. Sorted p: 0.002 ≤ 0.0083 ✓, 0.009 ≤ 0.0167 ✓, 0.020 ≤ 0.025 ✓, 0.040 > 0.033 ✗, 0.30 ✗, 0.80 ✗. The largest passing rank is 3, so reject the three smallest. (Bonferroni, p < 0.0083, rejects only 0.002.)</details>
3. Why are Poisson p-values anti-conservative for biological replicates?
   <details><summary>Answer</summary>The Poisson assumes Var = μ, but biological replicates have Var = μ + αμ². The Poisson standard error $\sqrt{[(X^\top WX)^{-1}]_{11}}$ with $W = \mathrm{diag}(\mu)$ is therefore too small, |z| too large and p too small; in Section 3.4 almost half of the truly unchanged genes had p < 0.05.</details>
4. A gene has true counts mean μ = 400 and dispersion α = 0.04. What are its Poisson and NB standard deviations, and what is the biological coefficient of variation?
   <details><summary>Answer</summary>Poisson: √400 = 20. NB: √(400 + 0.04·400²) = √(400 + 6400) = √6800 ≈ 82. BCV = √α = 0.2, i.e. replicates differ by about 20 % in true expression — at this depth biological noise dominates counting noise.</details>
5. In a pseudobulk analysis of monocytes from 4 control and 4 stimulated donors, with 300 cells per donor, how many replicates does the test have, and why is that the right number?
   <details><summary>Answer</summary>Eight (4 + 4) — one per donor. Conditions were assigned to donors' samples, and cells of one donor share its genetic and environmental effects, so they are not independent; the variability that matters for generalising to new people is between donors.</details>
6. A gene has estimated log2 fold change −1 (treated vs control). Interpret it, and give the natural-log coefficient β₁.
   <details><summary>Answer</summary>Treated samples express the gene at half the control level (2⁻¹ = 0.5), after size-factor normalisation. β₁ = −1 · ln 2 ≈ −0.693.</details>
7. What does the Cox–Reid adjustment correct, and what Level 1 idea is it analogous to?
   <details><summary>Answer</summary>The downward bias of the ML dispersion, which treats the fitted means as known although p coefficients were estimated from the same n samples. It is analogous to dividing the sample variance by N − 1 instead of N (Module 05).</details>
8. In `glm_irls`, what are `W` and `z`, and why does `W` change from `mu` to `mu / (1 + a * mu)` for the NB?
   <details><summary>Answer</summary>`W` holds the IRLS weights (inverse variance of each observation on the working scale) and `z` the working response η − offset + (y − μ)/μ. For the Poisson the variance of y is μ, giving weight μ; for the NB the variance is μ + αμ², and the weight becomes μ²/(μ + αμ²) = μ/(1 + αμ): high counts, which carry relatively more biological noise, are down-weighted.</details>

## 🧠 Questions & discussion

1. Our pipeline was slightly liberal (realised FDR a bit above 5 %) even after shrinkage. List three modelling choices that could cause this and how you would test each one with the simulator.
2. A collaborator has 2 control and 2 treated mice and wants a list of DE genes 'with FDR < 1 %'. What would you tell them, using what you measured in Section 6?
3. Median-of-ratios normalisation assumes most genes do not change. Describe a biological situation where this fails, and how you would design the experiment to detect or correct it.
4. Pseudobulk throws away the within-donor information of hundreds of cells. When might a mixed model with a random donor effect be preferable, and what would it estimate that pseudobulk cannot?
5. Differential-expression lists are often interpreted as 'the genes regulated by the treatment'. What else can make a gene appear DE in a tissue sample (think of cell-type composition), and how could single-cell data help?
6. Should a published DE analysis share its count matrix and design? Discuss the reproducibility benefits and the privacy limits for human data.

## 🚀 Bonus — going further

- **A Bayesian NB GLM with MCMC.** Write the log-posterior of $(\beta_0, \beta_1, \log\alpha)$ for one gene with weak normal priors and sample it with the random-walk Metropolis–Hastings of [Module 21](21_bayesian_learning_mcmc.ipynb). Compare the 95 % credible interval of $\beta_1$ with the Wald interval $\hat\beta_1 \pm 1.96\,\mathrm{SE}$.
- **Zero inflation.** Simulate counts with an extra 30 % structural zeros, fit an NB and a zero-inflated NB (mixture of a point mass at 0 and an NB, fitted by EM — [Module 13](13_mixture_models_em.ipynb)), and compare with the UMI-count conclusion of Svensson (2020).
- **LRT for a three-level factor.** Simulate control, low dose and high dose groups; test 'any dose effect' with an LRT on two degrees of freedom (full model with two dose columns vs intercept only).
- **Fold-change shrinkage.** Add a normal prior on $\beta_1$ with variance estimated from all genes (empirical Bayes again) and compare the ranking of genes by shrunken vs unshrunken log2 fold change against the truth.
- **Try the real tools.** Install PyDESeq2 (Python) or DESeq2/edgeR (R), export a simulated count matrix and design, and compare their size factors, dispersions and adjusted p-values with our mini-DESeq2.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 24 — Generalised linear models for count data            Date:
Biological question: which genes change after stimulation (bulk RNA-seq; monocytes in a Kang-like scRNA-seq design)?
Data (SIMULATED): simulate_bulk_counts (2,000 genes, 4 vs 4; truth: DE genes, log2FC, dispersions, size factors);
                  simulate_scrna (3,000 cells, 6 donors; truth: true_logfc per cell type)
Method: NB GLM with offsets fitted by IRLS; median-of-ratios size factors; Cox–Reid dispersions shrunk to a trend
        (empirical Bayes); Wald/LRT; Benjamini–Hochberg; pseudobulk; covariates and interactions
Key equation: log mu_gi = log s_i + x_iᵀ beta_g;  Var(y) = mu + alpha mu²;  beta ← (XᵀWX)⁻¹XᵀWz,  W = mu/(1 + alpha mu)
Key parameters: replicates per group; dispersion prior variance (floor 0.25); BH level q; expression filter
Ground-truth check result: realised FDR / power with gene-wise vs shrunken dispersions: … ; cell-level vs pseudobulk: …
What changed when I modified …: (e.g. n_per_group, donor_effect, n_dominant, batch_effect)
One failure mode: (e.g. Poisson p-values; total-count normalisation under composition bias; cells as replicates)
Connection to research (paper/tool): DESeq2 (Love et al. 2014), edgeR, limma-voom; Squair et al. 2021 (pseudobulk)
```

## 📚 References

- McCullagh P., Nelder J. A. (1989) *Generalized Linear Models*, 2nd ed. Chapman & Hall.
- Nelder J. A., Wedderburn R. W. M. (1972) Generalized linear models. *JRSS-A* 135:370–384.
- Anders S., Huber W. (2010) Differential expression analysis for sequence count data. *Genome Biology* 11:R106.
- Robinson M. D., McCarthy D. J., Smyth G. K. (2010) edgeR: a Bioconductor package for differential expression analysis of digital gene expression data. *Bioinformatics* 26:139–140.
- Robinson M. D., Oshlack A. (2010) A scaling normalization method for differential expression analysis of RNA-seq data. *Genome Biology* 11:R25.
- Love M. I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biology* 15:550.
- Law C. W., Chen Y., Shi W., Smyth G. K. (2014) voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology* 15:R29.
- Cox D. R., Reid N. (1987) Parameter orthogonality and approximate conditional inference. *JRSS-B* 49:1–39.
- Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate: a practical and powerful approach to multiple testing. *JRSS-B* 57:289–300.
- Storey J. D. (2002) A direct approach to false discovery rates. *JRSS-B* 64:479–498.
- Storey J. D., Tibshirani R. (2003) Statistical significance for genomewide studies. *PNAS* 100:9440–9445.
- Bourgon R., Gentleman R., Huber W. (2010) Independent filtering increases detection power for high-throughput experiments. *PNAS* 107:9546–9551.
- Schurch N. J. *et al.* (2016) How many biological replicates are needed in an RNA-seq experiment and which differential expression tool should you use? *RNA* 22:839–851.
- Kang H. M. *et al.* (2018) Multiplexed droplet single-cell RNA-sequencing using natural genetic variation. *Nature Biotechnology* 36:89–94.
- Squair J. W. *et al.* (2021) Confronting false discoveries in single-cell differential expression. *Nature Communications* 12:5692.
- Crowell H. L. *et al.* (2020) muscat detects subpopulation-specific state transitions from multi-sample multi-condition single-cell transcriptomics data. *Nature Communications* 11:6077.
- Hafemeister C., Satija R. (2019) Normalization and variance stabilization of single-cell RNA-seq data using regularized negative binomial regression. *Genome Biology* 20:296.
- Lause J., Berens P., Kobak D. (2021) Analytic Pearson residuals for normalization of single-cell RNA-seq UMI data. *Genome Biology* 22:258.
- Svensson V. (2020) Droplet scRNA-seq is not zero-inflated. *Nature Biotechnology* 38:147–150.
- Li W. *et al.* (2014) MAGeCK enables robust identification of essential genes from genome-scale CRISPR/Cas9 knockout screens. *Genome Biology* 15:554.

## ➡️ Next

The NB likelihood will follow us. [Module 27 · Variational inference and VAEs](27_variational_inference_vae.ipynb)
replaces the linear predictor of our GLM by a neural network and adds a latent variable per cell — a mini-scVI — and
[Module 30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb) uses Pearson residuals and pseudobulk from this module
in a complete single-cell analysis.